In [ ]:
"""
Representar um grafo em Python
"""

grafo = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [], 
    'E': ['F'], 
    'F': []
}

print(grafo)

print('\n')

for node, neighbors in grafo.items():
    print(f"{node} -> {', '.join(neighbors) if neighbors else 'Nada'}")

In [ ]:
"""
Transformar grafo em árvore binária
"""

class TreeNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

def convert_to_binary_tree(graph, root_value):
    if root_value not in graph:
        return None

    root = TreeNode(root_value)
    children = graph[root_value]

    if len(children) > 0:
        root.left = convert_to_binary_tree(graph, children[0])  # Primeiro filho -> Esquerda
    if len(children) > 1:
        root.right = convert_to_binary_tree(graph, children[1])  # Segundo filho -> Direita

    return root

def print_binary_tree(node, level=0, prefix="Root: "):
    """ Exibe a árvore de forma estruturada """
    if node is not None:
        print(" " * (level * 4) + prefix + str(node.value))
        print_binary_tree(node.left, level + 1, "L--> ")
        print_binary_tree(node.right, level + 1, "R--> ")

# Grafo de entrada
grafo = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': ['F'],
    'F': []
}

# Converter e mostrar a árvore
binary_tree_root = convert_to_binary_tree(grafo, 'A')
print_binary_tree(binary_tree_root)

In [ ]:
"""
Grafo com pesos
"""

import csv
from collections import defaultdict

# Constrói o grafo no formato pedido: node -> [[neighbor, peso], ...]
def build_grafo(csv_path, weight_col, undirected=True):
    adj = defaultdict(dict)
    nodes = set()

    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            u = row['source'].strip()
            v = row['target'].strip()
            nodes.add(u)
            nodes.add(v)

            w = float(row[weight_col])
            adj[u][v] = w
            if undirected:
                adj[v][u] = w

    return {n: [[m, adj[n][m]] for m in sorted(adj[n].keys())] for n in sorted(nodes)}

grafo_distancia = build_grafo('ia_tp1 - edges.csv', 'distance(km)', undirected=True)
grafo = grafo_distancia

print(grafo)

print('\n')

for node, neighbors in grafo.items():
    connections = ", ".join([f"{neighbor}({cost})" for neighbor, cost in neighbors])
    print(f"{node} -> {connections if connections else 'Nada'}")

In [ ]:
"""
Converter grafo com custo em árvore 
"""

class TreeNode:
    def __init__(self, value, weight=None):
        self.value = value
        self.weight = weight  # Peso da aresta que leva a este nó
        self.left = None
        self.right = None

def convert_to_binary_tree(graph, root_value, weight=None):
    if root_value not in graph:
        return None

    root = TreeNode(root_value, weight)
    children = graph[root_value]

    if len(children) > 0:
        root.left = convert_to_binary_tree(graph, children[0][0], children[0][1])  # Primeiro filho -> Esquerda
    if len(children) > 1:
        root.right = convert_to_binary_tree(graph, children[1][0], children[1][1])  # Segundo filho -> Direita

    return root

def print_binary_tree(node, level=0, prefix="Root: "):
    """ Exibe a árvore de forma estruturada, incluindo pesos das arestas """
    if node is not None:
        weight_info = f" (w={node.weight})" if node.weight is not None else ""
        print(" " * (level * 4) + prefix + str(node.value) + weight_info)
        print_binary_tree(node.left, level + 1, "L--> ")
        print_binary_tree(node.right, level + 1, "R--> ")

# Grafo ponderado de entrada
grafo = grafo_distancia

# Converter e mostrar a árvore
binary_tree_root = convert_to_binary_tree(grafo, next(iter(grafo)))
print_binary_tree(binary_tree_root)

In [ ]:
"""
Procura em extensão numa árvore a partir de um grafo
"""

from collections import deque

class No:
    def __init__(self, valor):
        self.valor = valor
        self.esquerda = None
        self.direita = None

# Função para construir a árvore binária a partir do grafo
def construir_arvore_binaria(grafo, raiz):
    raiz_no = No(raiz)
    
    # Função recursiva para criar os filhos
    def construir_filho(no, vizinhos):
        if not vizinhos:
            return
        
        if no.esquerda is None and len(vizinhos) > 0:
            filho_valor = vizinhos[0][0]
            no.esquerda = No(filho_valor)
            construir_filho(no.esquerda, grafo.get(filho_valor, []))
        
        if no.direita is None and len(vizinhos) > 1:
            filho_valor = vizinhos[1][0]
            no.direita = No(filho_valor)
            construir_filho(no.direita, grafo.get(filho_valor, []))
    
    construir_filho(raiz_no, grafo.get(raiz, []))
    return raiz_no

# Função de procura em extensão (BFS)
def procura_em_extensao(raiz, objetivo):
    # Fila para armazenar os nós a serem visitados
    fila = deque([(raiz, [raiz.valor])])  # Armazena o nó e o caminho até ele
    nos_expandidos = []  # Lista para armazenar os nós visitados
    
    while fila:
        no_atual, caminho = fila.popleft()
        
        # Adiciona o nó atual à lista de nós expandidos
        nos_expandidos.append(no_atual.valor)
        
        # Se encontramos o objetivo, retorna o caminho
        if no_atual.valor == objetivo:
            return caminho, nos_expandidos
        
        # Adicionar filhos à fila (se existirem)
        if no_atual.esquerda:
            fila.append((no_atual.esquerda, caminho + [no_atual.esquerda.valor]))
        
        if no_atual.direita:
            fila.append((no_atual.direita, caminho + [no_atual.direita.valor]))
    
    # Caso não encontre o objetivo
    return None, nos_expandidos

# Função para pedir os nós inicial e destino
def pedir_nos():
    inicial = input("Informe o nó inicial: ")
    destino = input("Informe o nó destino: ")
    return inicial, destino

# Grafo fornecido
grafo = { 
    'A': [['B'], ['C']],  # A -> B (esquerdo), A -> C (direito)
    'B': [['D'], ['E']],
    'C': [['F']],
    'D': [], 
    'E': [['F']], 
    'F': []
}

# Pedir os nós inicial e destino
inicial, destino = pedir_nos()

# Construir a árvore binária a partir do grafo
raiz_arvore = construir_arvore_binaria(grafo, inicial)

# Executar a procura em extensão
caminho, nos_expandidos = procura_em_extensao(raiz_arvore, destino)

# Exibir o resultado
if caminho:
    print(f"Caminho encontrado: {caminho}")
else:
    print(f"Não foi possível encontrar o caminho de {inicial} até {destino}.")

# Mostrar os nós expandidos
print(f"Nos expandidos: {nos_expandidos}")

In [ ]:
"""
Usar um agente na procura em extensão
"""

from collections import deque

class Agent:
    def __init__(self, graph, start, goal):
        self.graph = graph  # Representação do ambiente
        self.start = start  # Estado inicial
        self.goal = goal    # Estado objetivo

    def act(self):
        """Define como o agente age. Subclasses irão implementar este método."""
        raise NotImplementedError("Este método deve ser implementado por subclasses.")


class BFSAgent(Agent):
    def act(self):
        queue = deque([self.start])
        visited = set()
        parent = {self.start: None}

        while queue:
            current = queue.popleft()
            
            if current == self.goal:
                # Reconstrução do caminho
                path = []
                while current:
                    path.append(current)
                    current = parent[current]
                return path[::-1]

            visited.add(current)

            for neighbor in self.graph[current]:
                if neighbor not in visited and neighbor not in queue:
                    queue.append(neighbor)
                    parent[neighbor] = current

        return None  # Objetivo não encontrado

# Exemplo de uso
grafo = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [], 
    'E': ['F'], 
    'F': []
}

agent = BFSAgent(grafo, 'A', 'F')
print(agent.act())  

In [ ]:
"""
Procura em perfundidade numa árvore a partir de um grafo
"""

class No:
    def __init__(self, valor):
        self.valor = valor
        self.esquerda = None
        self.direita = None

# Função para construir a árvore binária a partir do grafo
def construir_arvore_binaria(grafo, raiz):
    raiz_no = No(raiz)
    
    # Função recursiva para criar os filhos
    def construir_filho(no, vizinhos):
        if not vizinhos:
            return
        
        if no.esquerda is None and len(vizinhos) > 0:
            filho_valor = vizinhos[0][0]
            no.esquerda = No(filho_valor)
            construir_filho(no.esquerda, grafo.get(filho_valor, []))
        
        if no.direita is None and len(vizinhos) > 1:
            filho_valor = vizinhos[1][0]
            no.direita = No(filho_valor)
            construir_filho(no.direita, grafo.get(filho_valor, []))
    
    construir_filho(raiz_no, grafo.get(raiz, []))
    return raiz_no

# Função de procura em profundidade (DFS)
def procura_em_profundidade(raiz, objetivo):
    pilha = [(raiz, [raiz.valor])]  # Pilha de nós a visitar (contém o nó e o caminho até ele)
    nos_expandidos = []  # Lista para armazenar os nós expandidos
    
    while pilha:
        no_atual, caminho = pilha.pop()  # Retira o nó mais profundo da pilha
        
        # Adiciona o nó atual à lista de nós expandidos
        nos_expandidos.append(no_atual.valor)
        
        # Se encontramos o objetivo, retorna o caminho
        if no_atual.valor == objetivo:
            return caminho, nos_expandidos
        
        # Adicionar filhos à pilha (se existirem), explorando a direita e depois a esquerda
        if no_atual.direita:
            pilha.append((no_atual.direita, caminho + [no_atual.direita.valor]))
        
        if no_atual.esquerda:
            pilha.append((no_atual.esquerda, caminho + [no_atual.esquerda.valor]))
    
    # Caso não encontre o objetivo
    return None, nos_expandidos

# Função para pedir os nós inicial e destino
def pedir_nos():
    inicial = input("Informe o nó inicial: ")
    destino = input("Informe o nó destino: ")
    return inicial, destino

# Grafo fornecido
grafo = { 
    'A': [['B'], ['C']],  # A -> B (esquerdo), A -> C (direito)
    'B': [['D'], ['E']],
    'C': [['F']],
    'D': [], 
    'E': [['F']], 
    'F': []
}

# Pedir os nós inicial e destino
inicial, destino = pedir_nos()

# Construir a árvore binária a partir do grafo
raiz_arvore = construir_arvore_binaria(grafo, inicial)

# Executar a procura em profundidade
caminho, nos_expandidos = procura_em_profundidade(raiz_arvore, destino)

# Exibir o resultado
if caminho:
    print(f"Caminho encontrado: {caminho}")
else:
    print(f"Não foi possível encontrar o caminho de {inicial} até {destino}.")

# Mostrar os nós expandidos
print(f"Nos expandidos: {nos_expandidos}")

In [ ]:
"""
Procura de custo uniforme numa árvore a partir de um grafo
"""

import heapq  # Usamos o heapq para implementar a fila de prioridade

class No:
    def __init__(self, valor):
        self.valor = valor
        self.esquerda = None
        self.direita = None

# Função para construir a árvore binária a partir do grafo
def construir_arvore_binaria(grafo, raiz):
    raiz_no = No(raiz)
    
    # Função recursiva para criar os filhos
    def construir_filho(no, vizinhos):
        if not vizinhos:
            return
        
        if no.esquerda is None and len(vizinhos) > 0:
            filho_valor = vizinhos[0][0]
            no.esquerda = No(filho_valor)
            construir_filho(no.esquerda, grafo.get(filho_valor, []))
        
        if no.direita is None and len(vizinhos) > 1:
            filho_valor = vizinhos[1][0]
            no.direita = No(filho_valor)
            construir_filho(no.direita, grafo.get(filho_valor, []))
    
    construir_filho(raiz_no, grafo.get(raiz, []))
    return raiz_no

# Função de procura de custo uniforme (UCS)
def procura_de_custo_uniforme(grafo, inicial, destino):
    # Fila de prioridade para armazenar os nós a serem expandidos (custo, nó, caminho)
    fila = []
    heapq.heappush(fila, (0, inicial, [inicial]))  # (custo acumulado, nó, caminho)
    
    # Conjunto para armazenar os nós visitados
    nos_visitados = set()
    
    # Lista para armazenar os nós expandidos
    nos_expandidos = []
    
    while fila:
        custo_atual, no_atual, caminho_atual = heapq.heappop(fila)
        
        # Se já visitamos o nó, ignoramos
        if no_atual in nos_visitados:
            continue
        
        # Marca o nó como visitado
        nos_visitados.add(no_atual)
        nos_expandidos.append(no_atual)
        
        # Se encontramos o nó de destino, retorna o caminho e o custo
        if no_atual == destino:
            return caminho_atual, custo_atual, nos_expandidos
        
        # Explora os vizinhos
        for vizinho, custo in grafo.get(no_atual, []):
            if vizinho not in nos_visitados:
                novo_custo = custo_atual + custo
                novo_caminho = caminho_atual + [vizinho]
                heapq.heappush(fila, (novo_custo, vizinho, novo_caminho))
    
    # Caso não encontre o objetivo
    return None, None, nos_expandidos

# Função para pedir os nós inicial e destino
def pedir_nos():
    inicial = input("Informe o nó inicial: ")
    destino = input("Informe o nó destino: ")
    return inicial, destino

# Grafo fornecido
grafo = grafo_distancia

# Pedir os nós inicial e destino
inicial, destino = pedir_nos()

# Executar a procura de custo uniforme
caminho, custo, nos_expandidos = procura_de_custo_uniforme(grafo, inicial, destino)

# Exibir o resultado
if caminho:
    print(f"Caminho encontrado: {caminho}")
    print(f"Custo total: {custo}")
else:
    print(f"Não foi possível encontrar o caminho de {inicial} até {destino}.")

# Mostrar os nós expandidos
print(f"Nos expandidos: {nos_expandidos}")

In [ ]:
"""
Procura sofrega (greedy) numa árvore a partir de um grafo
"""

import heapq  # Usamos o heapq para implementar a fila de prioridade

class No:
    def __init__(self, valor):
        self.valor = valor
        self.esquerda = None
        self.direita = None

# Função para construir a árvore binária a partir do grafo
def construir_arvore_binaria(grafo, raiz):
    raiz_no = No(raiz)
    
    # Função recursiva para criar os filhos
    def construir_filho(no, vizinhos):
        if not vizinhos:
            return
        
        if no.esquerda is None and len(vizinhos) > 0:
            filho_valor = vizinhos[0][0]
            no.esquerda = No(filho_valor)
            construir_filho(no.esquerda, grafo.get(filho_valor, []))
        
        if no.direita is None and len(vizinhos) > 1:
            filho_valor = vizinhos[1][0]
            no.direita = No(filho_valor)
            construir_filho(no.direita, grafo.get(filho_valor, []))
    
    construir_filho(raiz_no, grafo.get(raiz, []))
    return raiz_no

# Função de procura greedy com heurística (Best-First Search)
def procura_greedy(grafo, inicial, destino, heuristica):
    # Fila de prioridade (para a busca greedy, com base na heurística)
    fila = []
    heapq.heappush(fila, (heuristica(inicial, destino), inicial, [inicial]))  # (heurística, nó, caminho)
    
    # Conjunto para armazenar os nós visitados
    nos_visitados = set()
    
    # Lista para armazenar os nós expandidos
    nos_expandidos = []
    
    while fila:
        _, no_atual, caminho_atual = heapq.heappop(fila)
        
        # Se já visitamos o nó, ignoramos
        if no_atual in nos_visitados:
            continue
        
        # Marca o nó como visitado
        nos_visitados.add(no_atual)
        nos_expandidos.append(no_atual)
        
        # Se encontramos o nó de destino, retorna o caminho
        if no_atual == destino:
            return caminho_atual, nos_expandidos
        
        # Explora os vizinhos (ordem de exploração: esquerda primeiro)
        for vizinho in grafo.get(no_atual, []):
            vizinho_valor = vizinho[0]  # Nome do vizinho
            if vizinho_valor not in nos_visitados:
                # Na busca greedy, usamos a heurística para decidir a ordem de expansão
                heapq.heappush(fila, (heuristica(vizinho_valor, destino), vizinho_valor, caminho_atual + [vizinho_valor]))
    
    # Caso não encontre o objetivo
    return None, nos_expandidos

# Heurística: Distância até o destino (número de saltos entre os nós)
def heuristica(no, destino):
    # Um exemplo simples de heurística seria a "distância em saltos" (número de arestas)
    # Considerando que a heurística é baseada na posição lexicográfica para simplificação
    # No caso, retornamos a diferença de "ordem" entre os nós
    heuristica_valor = abs(ord(no[0]) - ord(destino[0]))  # diferença dos valores das letras
    return heuristica_valor

# Função para pedir os nós inicial e destino
def pedir_nos():
    inicial = input("Informe o nó inicial: ")
    destino = input("Informe o nó destino: ")
    return inicial, destino

# Grafo fornecido
grafo = { 
    'A': [['B'], ['C']],  # A -> B (esquerdo), A -> C (direito)
    'B': [['D'], ['E']],
    'C': [['F']],
    'D': [], 
    'E': [['F']], 
    'F': []
}

# Pedir os nós inicial e destino
inicial, destino = pedir_nos()

# Construir a árvore binária a partir do grafo
raiz_arvore = construir_arvore_binaria(grafo, inicial)

# Executar a procura greedy com a heurística
caminho, nos_expandidos = procura_greedy(grafo, inicial, destino, heuristica)

# Exibir o resultado
if caminho:
    print(f"Caminho encontrado: {caminho}")
else:
    print(f"Não foi possível encontrar o caminho de {inicial} até {destino}.")

# Mostrar os nós expandidos
print(f"Nos expandidos: {nos_expandidos}")

In [ ]:
"""
Procura A* numa árvore a partir de um grafo
"""

import heapq  # Usamos o heapq para implementar a fila de prioridade

class No:
    def __init__(self, valor):
        self.valor = valor
        self.esquerda = None
        self.direita = None

# Função para construir a árvore binária a partir do grafo
def construir_arvore_binaria(grafo, raiz):
    raiz_no = No(raiz)
    
    # Função recursiva para criar os filhos
    def construir_filho(no, vizinhos):
        if not vizinhos:
            return
        
        if no.esquerda is None and len(vizinhos) > 0:
            filho_valor = vizinhos[0][0]
            no.esquerda = No(filho_valor)
            construir_filho(no.esquerda, grafo.get(filho_valor, []))
        
        if no.direita is None and len(vizinhos) > 1:
            filho_valor = vizinhos[1][0]
            no.direita = No(filho_valor)
            construir_filho(no.direita, grafo.get(filho_valor, []))
    
    construir_filho(raiz_no, grafo.get(raiz, []))
    return raiz_no

# Função de procura A* com custo acumulado e heurística
def procura_a_star(grafo, inicial, destino, heuristica):
    # Fila de prioridade (usada para A*, com f(n) = g(n) + h(n))
    fila = []
    heapq.heappush(fila, (heuristica(inicial, destino), 0, inicial, [inicial]))  # (f(n), g(n), nó, caminho)
    
    # Conjunto para armazenar os nós visitados
    nos_visitados = set()
    
    # Lista para armazenar os nós expandidos
    nos_expandidos = []
    
    while fila:
        _, g_atual, no_atual, caminho_atual = heapq.heappop(fila)
        
        # Se já visitamos o nó, ignoramos
        if no_atual in nos_visitados:
            continue
        
        # Marca o nó como visitado
        nos_visitados.add(no_atual)
        nos_expandidos.append(no_atual)
        
        # Se encontramos o nó de destino, retorna o caminho
        if no_atual == destino:
            return caminho_atual, nos_expandidos
        
        # Explora os vizinhos (ordem de exploração: esquerda primeiro)
        for vizinho in grafo.get(no_atual, []):
            vizinho_valor = vizinho[0]  # Nome do vizinho
            if vizinho_valor not in nos_visitados:
                # Calcula o custo acumulado g(n) para o vizinho
                g_vizinho = g_atual + 1  # Assume que a distância entre nós é 1 (se for diferente, modifique aqui)
                # Calcula a heurística h(n) para o vizinho
                f_vizinho = g_vizinho + heuristica(vizinho_valor, destino)
                heapq.heappush(fila, (f_vizinho, g_vizinho, vizinho_valor, caminho_atual + [vizinho_valor]))
    
    # Caso não encontre o objetivo
    return None, nos_expandidos

# Heurística: Distância até o destino (número de saltos entre os nós)
def heuristica(no, destino):
    # Um exemplo simples de heurística seria a "distância em saltos" (número de arestas)
    # Considerando que a heurística é baseada na posição lexicográfica para simplificação
    # No caso, retornamos a diferença de "ordem" entre os nós
    heuristica_valor = abs(ord(no[0]) - ord(destino[0]))  # diferença dos valores das letras
    return heuristica_valor

# Função para pedir os nós inicial e destino
def pedir_nos():
    inicial = input("Informe o nó inicial: ")
    destino = input("Informe o nó destino: ")
    return inicial, destino

# Grafo fornecido
grafo = { 
    'A': [['B'], ['C']],  # A -> B (esquerdo), A -> C (direito)
    'B': [['D'], ['E']],
    'C': [['F']],
    'D': [], 
    'E': [['F']], 
    'F': []
}

# Pedir os nós inicial e destino
inicial, destino = pedir_nos()

# Construir a árvore binária a partir do grafo
raiz_arvore = construir_arvore_binaria(grafo, inicial)

# Executar a procura A* com a heurística
caminho, nos_expandidos = procura_a_star(grafo, inicial, destino, heuristica)

# Exibir o resultado
if caminho:
    print(f"Caminho encontrado: {caminho}")
else:
    print(f"Não foi possível encontrar o caminho de {inicial} até {destino}.")

# Mostrar os nós expandidos
print(f"Nos expandidos: {nos_expandidos}")

In [ ]:
"""
Comparar a performance dos algoritmos de procura num grafo
"""

import time
import heapq

# Função para medir o tempo de execução e contar os nós expandidos
def medir_tempo_e_nos_expandidos(procura_funcao, *args):
    start_time = time.time()
    caminho, nos_expandidos = procura_funcao(*args)
    end_time = time.time()
    tempo_execucao = end_time - start_time
    return caminho, nos_expandidos, tempo_execucao, len(nos_expandidos)

# Função de procura em profundidade
def procura_profundidade(grafo, inicial, destino):
    visitados = set()
    caminho = []
    nos_expandidos = []

    def dfs(no):
        if no in visitados:
            return False
        visitados.add(no)
        caminho.append(no)
        nos_expandidos.append(no)

        if no == destino:
            return True
        
        for vizinho in grafo.get(no, []):
            if dfs(vizinho[0]):
                return True
        
        caminho.pop()
        return False
    
    dfs(inicial)
    return caminho, nos_expandidos

# Função de procura em largura (extensão)
def procura_extensao(grafo, inicial, destino):
    fila = [inicial]
    visitados = set()
    caminho = []
    nos_expandidos = []
    
    while fila:
        no_atual = fila.pop(0)
        if no_atual in visitados:
            continue
        visitados.add(no_atual)
        caminho.append(no_atual)
        nos_expandidos.append(no_atual)
        
        if no_atual == destino:
            return caminho, nos_expandidos
        
        for vizinho in grafo.get(no_atual, []):
            fila.append(vizinho[0])
    
    return caminho, nos_expandidos

# Função de procura de custo uniforme
def procura_custo_uniforme(grafo, inicial, destino):
    fila = []
    heapq.heappush(fila, (0, inicial, [inicial]))
    visitados = set()
    nos_expandidos = []
    
    while fila:
        custo_atual, no_atual, caminho_atual = heapq.heappop(fila)
        
        if no_atual in visitados:
            continue
        visitados.add(no_atual)
        nos_expandidos.append(no_atual)
        
        if no_atual == destino:
            return caminho_atual, nos_expandidos
        
        for vizinho in grafo.get(no_atual, []):
            custo_vizinho = custo_atual + vizinho[1]  # Considerando custo entre nós
            heapq.heappush(fila, (custo_vizinho, vizinho[0], caminho_atual + [vizinho[0]]))
    
    return [], nos_expandidos

# Função de procura sofrega
def procura_sofrega(grafo, inicial, destino, heuristica):
    fila = []
    heapq.heappush(fila, (heuristica(inicial, destino), inicial, [inicial]))
    visitados = set()
    nos_expandidos = []
    
    while fila:
        _, no_atual, caminho_atual = heapq.heappop(fila)
        
        if no_atual in visitados:
            continue
        visitados.add(no_atual)
        nos_expandidos.append(no_atual)
        
        if no_atual == destino:
            return caminho_atual, nos_expandidos
        
        for vizinho in grafo.get(no_atual, []):
            heapq.heappush(fila, (heuristica(vizinho[0], destino), vizinho[0], caminho_atual + [vizinho[0]]))
    
    return [], nos_expandidos

# Função de procura A* com heurística
def procura_a_star(grafo, inicial, destino, heuristica):
    fila = []
    heapq.heappush(fila, (heuristica(inicial, destino), 0, inicial, [inicial]))  # (f(n), g(n), nó, caminho)
    visitados = set()
    nos_expandidos = []
    
    while fila:
        _, g_atual, no_atual, caminho_atual = heapq.heappop(fila)
        
        if no_atual in visitados:
            continue
        visitados.add(no_atual)
        nos_expandidos.append(no_atual)
        
        if no_atual == destino:
            return caminho_atual, nos_expandidos
        
        for vizinho in grafo.get(no_atual, []):
            g_vizinho = g_atual + 1  # Assume que a distância entre nós é 1 (pode ser alterado)
            f_vizinho = g_vizinho + heuristica(vizinho[0], destino)
            heapq.heappush(fila, (f_vizinho, g_vizinho, vizinho[0], caminho_atual + [vizinho[0]]))
    
    return [], nos_expandidos

# Heurística para a procura sofrega e A*
def heuristica(no, destino):
    # Usando uma heurística simples baseada na diferença lexicográfica dos nós
    return abs(ord(no[0]) - ord(destino[0]))

# Grafo fornecido
grafo = grafo_distancia

# Pedir os nós inicial e destino
inicial = input("Informe o nó inicial: ")
destino = input("Informe o nó destino: ")

# Comparar os algoritmos
algoritmos = {
    "procura em Extensão": procura_extensao,
    "procura em Profundidade": procura_profundidade,
    "procura de Custo Uniforme": procura_custo_uniforme,
    "procura sofrega": procura_sofrega,
    "procura A*": procura_a_star
}

for nome, algoritmo in algoritmos.items():
    if nome == "procura sofrega" or nome == "procura A*":
        caminho, nos_expandidos, tempo, num_nos_expandidos = medir_tempo_e_nos_expandidos(algoritmo, grafo, inicial, destino, heuristica)
    else:
        caminho, nos_expandidos, tempo, num_nos_expandidos = medir_tempo_e_nos_expandidos(algoritmo, grafo, inicial, destino)
    
    print(f"\n{nome}:")
    print(f"Caminho: {caminho}")
    print(f"Nos Expandidos: {nos_expandidos}")
    print(f"Numero de Nos Expandidos: {num_nos_expandidos}")
    print(f"Tempo de Execução: {tempo:.6f} segundos")

In [ ]:
"""
Comparação dos algoritmos de procura no problema do Mapa da Roménia
"""

import heapq
import time

# Função para medir o tempo de execução e contar os nós expandidos
def medir_tempo_e_nos_expandidos(procura_funcao, *args):
    start_time = time.time()
    caminho, nos_expandidos = procura_funcao(*args)
    end_time = time.time()
    tempo_execucao = end_time - start_time
    return caminho, nos_expandidos, tempo_execucao, len(nos_expandidos)

# Função para verificar se alguém chegou ao objetivo
def objetivo(estado, estado_objetivo):
    return estado == estado_objetivo

# Função para gerar os vizinhos de uma cidade
def movimentos(estado, grafo):
    return grafo[estado] if estado in grafo else []

# procura em largura (extensão)
def procura_extensao(estado_inicial, estado_objetivo, grafo):
    fila = [estado_inicial]
    visitados = set()
    caminho = []
    nos_expandidos = []

    while fila:
        estado_atual = fila.pop(0)
        if estado_atual in visitados:
            continue
        visitados.add(estado_atual)
        caminho.append(estado_atual)
        nos_expandidos.append(estado_atual)

        if objetivo(estado_atual, estado_objetivo):
            return caminho, nos_expandidos

        for prox_estado, _ in movimentos(estado_atual, grafo):
            fila.append(prox_estado)

    return caminho, nos_expandidos

# procura de profundidade (iterativa)
def procura_profundidade_iterativa(estado_inicial, estado_objetivo, grafo):
    limite = 0
    while True:
        caminho, nos_expandidos = procura_profundidade_limitada(estado_inicial, estado_objetivo, grafo, limite)
        if caminho:
            return caminho, nos_expandidos
        limite += 1

def procura_profundidade_limitada(estado_inicial, estado_objetivo, grafo, limite):
    pilha = [(estado_inicial, [estado_inicial])]
    visitados = set()
    nos_expandidos = []

    while pilha:
        estado_atual, caminho_atual = pilha.pop()
        if estado_atual in visitados:
            continue
        visitados.add(estado_atual)
        nos_expandidos.append(estado_atual)

        if objetivo(estado_atual, estado_objetivo):
            return caminho_atual, nos_expandidos

        if len(caminho_atual) < limite:
            for prox_estado, _ in movimentos(estado_atual, grafo):
                pilha.append((prox_estado, caminho_atual + [prox_estado]))

    return [], nos_expandidos

# procura de custo uniforme
def procura_custo_uniforme(estado_inicial, estado_objetivo, grafo):
    fila = []
    heapq.heappush(fila, (0, estado_inicial, [estado_inicial]))
    visitados = set()
    nos_expandidos = []

    while fila:
        custo_atual, estado_atual, caminho_atual = heapq.heappop(fila)

        if estado_atual in visitados:
            continue
        visitados.add(estado_atual)
        nos_expandidos.append(estado_atual)

        if objetivo(estado_atual, estado_objetivo):
            return caminho_atual, nos_expandidos

        for prox_estado, custo in movimentos(estado_atual, grafo):
            custo_vizinho = custo_atual + custo
            heapq.heappush(fila, (custo_vizinho, prox_estado, caminho_atual + [prox_estado]))

    return [], nos_expandidos

# Função de procura sofrega
def procura_sofrega(estado_inicial, estado_objetivo, heuristica, grafo):
    fila = []
    heapq.heappush(fila, (heuristica(estado_inicial, estado_objetivo), estado_inicial, [estado_inicial]))
    visitados = set()
    nos_expandidos = []

    while fila:
        _, estado_atual, caminho_atual = heapq.heappop(fila)

        if estado_atual in visitados:
            continue
        visitados.add(estado_atual)
        nos_expandidos.append(estado_atual)

        if objetivo(estado_atual, estado_objetivo):
            return caminho_atual, nos_expandidos

        for prox_estado, _ in movimentos(estado_atual, grafo):
            heapq.heappush(fila, (heuristica(prox_estado, estado_objetivo), prox_estado, caminho_atual + [prox_estado]))

    return [], nos_expandidos

# Função de procura A* com heurística
def procura_a_star(estado_inicial, estado_objetivo, heuristica, grafo):
    fila = []
    heapq.heappush(fila, (heuristica(estado_inicial, estado_objetivo), 0, estado_inicial, [estado_inicial]))
    visitados = set()
    nos_expandidos = []

    while fila:
        _, g_atual, estado_atual, caminho_atual = heapq.heappop(fila)

        if estado_atual in visitados:
            continue
        visitados.add(estado_atual)
        nos_expandidos.append(estado_atual)

        if objetivo(estado_atual, estado_objetivo):
            return caminho_atual, nos_expandidos

        for prox_estado, custo in movimentos(estado_atual, grafo):
            g_vizinho = g_atual + custo
            f_vizinho = g_vizinho + heuristica(prox_estado, estado_objetivo)
            heapq.heappush(fila, (f_vizinho, g_vizinho, prox_estado, caminho_atual + [prox_estado]))

    return [], nos_expandidos

# Função para calcular a heurística (distância de Manhattan, aqui simplificada como a distância de cada cidade em linha reta)
def heuristica(estado, estado_objetivo):
    # Distâncias simples para as cidades
    distancias = {
        'Arad': 366,
        'Zerind': 374,
        'Sibiu': 253,
        'Timisoara': 329,
        'Oradea': 380,
        'Fagaras': 178,
        'Rimnicu Vilcea': 193,
        'Lugoj': 244,
        'Cluj-Napoca': 280,
        'Bucharest': 0,
        'Craiova': 160,
        'Pitesti': 98,
        'Mehadia': 241,
        'Turda': 253,
        'Giurgiu': 77,
        'Drobeta': 242
    }
    return distancias.get(estado, 0)

# Exemplo de execução
estado_inicial = 'Arad'
estado_objetivo = 'Bucharest'
grafo_romenia = {
    'Arad': [['Zerind', 75], ['Sibiu', 140], ['Timisoara', 118]],
    'Zerind': [['Arad', 75], ['Oradea', 71]],
    'Sibiu': [['Arad', 140], ['Fagaras', 99], ['Rimnicu Vilcea', 80]],
    'Timisoara': [['Arad', 118], ['Lugoj', 111]],
    'Oradea': [['Zerind', 71], ['Cluj-Napoca', 151]],
    'Fagaras': [['Sibiu', 99], ['Bucharest', 211]],
    'Rimnicu Vilcea': [['Sibiu', 80], ['Craiova', 146], ['Pitesti', 97]],
    'Lugoj': [['Timisoara', 111], ['Mehadia', 70]],
    'Cluj-Napoca': [['Oradea', 151], ['Turda', 32]],
    'Bucharest': [['Fagaras', 211], ['Pitesti', 101], ['Giurgiu', 90]],
    'Craiova': [['Rimnicu Vilcea', 146], ['Pitesti', 138]],
    'Pitesti': [['Rimnicu Vilcea', 97], ['Bucharest', 101], ['Craiova', 138]],
    'Mehadia': [['Lugoj', 70], ['Drobeta', 75]],
    'Turda': [['Cluj-Napoca', 32], ['Sibiu', 140]],
    'Giurgiu': [['Bucharest', 90]],
    'Drobeta': [['Mehadia', 75], ['Craiova', 120]],
}

# Chamar todos os algoritmos de procura
def executar_procuras():
    resultados = {}

    # procura em Extensão
    caminho, nos_expandidos, tempo, num_nos_expandidos = medir_tempo_e_nos_expandidos(procura_extensao, estado_inicial, estado_objetivo, grafo_romenia)
    resultados['procura em Extensão'] = (caminho, nos_expandidos, tempo, num_nos_expandidos)

    # procura de Profundidade Iterativa
    caminho, nos_expandidos, tempo, num_nos_expandidos = medir_tempo_e_nos_expandidos(procura_profundidade_iterativa, estado_inicial, estado_objetivo, grafo_romenia)
    resultados['procura de Profundidade Iterativa'] = (caminho, nos_expandidos, tempo, num_nos_expandidos)

    # procura de Custo Uniforme
    caminho, nos_expandidos, tempo, num_nos_expandidos = medir_tempo_e_nos_expandidos(procura_custo_uniforme, estado_inicial, estado_objetivo, grafo_romenia)
    resultados['procura de Custo Uniforme'] = (caminho, nos_expandidos, tempo, num_nos_expandidos)

    # procura sofrega
    caminho, nos_expandidos, tempo, num_nos_expandidos = medir_tempo_e_nos_expandidos(procura_sofrega, estado_inicial, estado_objetivo, heuristica, grafo_romenia)
    resultados['procura sofrega'] = (caminho, nos_expandidos, tempo, num_nos_expandidos)

    # procura A*
    caminho, nos_expandidos, tempo, num_nos_expandidos = medir_tempo_e_nos_expandidos(procura_a_star, estado_inicial, estado_objetivo, heuristica, grafo_romenia)
    resultados['procura A*'] = (caminho, nos_expandidos, tempo, num_nos_expandidos)

    return resultados

# Exibindo os resultados
resultados = executar_procuras()
for nome, (caminho, nos_expandidos, tempo, num_nos_expandidos) in resultados.items():
    print(f"{nome}:")
    print(f"Caminho: {caminho}")
    print(f"Nos Expandidos: {nos_expandidos}")
    print(f"Numero de Nos Expandidos: {num_nos_expandidos}")
    print(f"Tempo de Execução: {tempo:.6f} segundos\n")

In [ ]:
"""
Comparação dos algoritmos de procura no problema Puzzle 8
"""

import time
import heapq

# Função para medir o tempo de execução e contar os nós expandidos
def medir_tempo_e_nos_expandidos(procura_funcao, *args):
    start_time = time.time()
    caminho, nos_expandidos = procura_funcao(*args)
    end_time = time.time()
    tempo_execucao = end_time - start_time
    return caminho, nos_expandidos, tempo_execucao, len(nos_expandidos)

# Movimentos possíveis no puzzle de 8
def movimentos(estado):
    pos = estado.index(0)
    pos_x, pos_y = divmod(pos, 3)
    movimentos = []
    
    direcoes = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # cima, baixo, esquerda, direita
    for dx, dy in direcoes:
        nova_x, nova_y = pos_x + dx, pos_y + dy
        if 0 <= nova_x < 3 and 0 <= nova_y < 3:
            novo_pos = nova_x * 3 + nova_y
            novo_estado = estado[:]
            novo_estado[pos], novo_estado[novo_pos] = novo_estado[novo_pos], novo_estado[pos]
            movimentos.append(novo_estado)
    
    return movimentos

# Função para verificar se o estado é o objetivo
def objetivo(estado, estado_objetivo):
    return estado == estado_objetivo

# Função de procura em profundidade (DFS) com verificação de visitados
def procura_profundidade(estado_inicial, estado_objetivo):
    # implementação iterativa em vez de recursiva
    pilha = [estado_inicial]
    visitados = set()
    caminho = []
    nos_expandidos = []
    
    while pilha:
        estado_atual = pilha.pop()
        
        # Se o estado já foi visitado, ignora
        if tuple(estado_atual) in visitados:
            continue
        
        # Marca o estado como visitado
        visitados.add(tuple(estado_atual))
        caminho.append(estado_atual)
        nos_expandidos.append(estado_atual)
        
        # Verifica se o estado atual é o objetivo
        if objetivo(estado_atual, estado_objetivo):
            return caminho, nos_expandidos
        
        # Expande os vizinhos e os coloca na pilha
        for prox_estado in movimentos(estado_atual):
            pilha.append(prox_estado)
    
    return caminho, nos_expandidos

# Função de procura em largura (extensão)
def procura_extensao(estado_inicial, estado_objetivo):
    fila = [estado_inicial]
    visitados = set()
    caminho = []
    nos_expandidos = []
    
    while fila:
        estado_atual = fila.pop(0)
        if tuple(estado_atual) in visitados:
            continue
        visitados.add(tuple(estado_atual))
        caminho.append(estado_atual)
        nos_expandidos.append(estado_atual)
        
        if objetivo(estado_atual, estado_objetivo):
            return caminho, nos_expandidos
        
        for prox_estado in movimentos(estado_atual):
            fila.append(prox_estado)
    
    return caminho, nos_expandidos

# Função de procura de custo uniforme
def procura_custo_uniforme(estado_inicial, estado_objetivo):
    fila = []
    heapq.heappush(fila, (0, estado_inicial, [estado_inicial]))
    visitados = set()
    nos_expandidos = []
    
    while fila:
        custo_atual, estado_atual, caminho_atual = heapq.heappop(fila)
        
        if tuple(estado_atual) in visitados:
            continue
        visitados.add(tuple(estado_atual))
        nos_expandidos.append(estado_atual)
        
        if objetivo(estado_atual, estado_objetivo):
            return caminho_atual, nos_expandidos
        
        for prox_estado in movimentos(estado_atual):
            custo_vizinho = custo_atual + 1
            heapq.heappush(fila, (custo_vizinho, prox_estado, caminho_atual + [prox_estado]))
    
    return [], nos_expandidos

# Função de procura sofrega
def procura_sofrega(estado_inicial, estado_objetivo, heuristica):
    fila = []
    heapq.heappush(fila, (heuristica(estado_inicial, estado_objetivo), estado_inicial, [estado_inicial]))
    visitados = set()
    nos_expandidos = []
    
    while fila:
        _, estado_atual, caminho_atual = heapq.heappop(fila)
        
        if tuple(estado_atual) in visitados:
            continue
        visitados.add(tuple(estado_atual))
        nos_expandidos.append(estado_atual)
        
        if objetivo(estado_atual, estado_objetivo):
            return caminho_atual, nos_expandidos
        
        for prox_estado in movimentos(estado_atual):
            heapq.heappush(fila, (heuristica(prox_estado, estado_objetivo), prox_estado, caminho_atual + [prox_estado]))
    
    return [], nos_expandidos

# Função de procura A* com heurística
def procura_a_star(estado_inicial, estado_objetivo, heuristica):
    fila = []
    heapq.heappush(fila, (heuristica(estado_inicial, estado_objetivo), 0, estado_inicial, [estado_inicial]))
    visitados = set()
    nos_expandidos = []
    
    while fila:
        _, g_atual, estado_atual, caminho_atual = heapq.heappop(fila)
        
        if tuple(estado_atual) in visitados:
            continue
        visitados.add(tuple(estado_atual))
        nos_expandidos.append(estado_atual)
        
        if objetivo(estado_atual, estado_objetivo):
            return caminho_atual, nos_expandidos
        
        for prox_estado in movimentos(estado_atual):
            g_vizinho = g_atual + 1
            f_vizinho = g_vizinho + heuristica(prox_estado, estado_objetivo)
            heapq.heappush(fila, (f_vizinho, g_vizinho, prox_estado, caminho_atual + [prox_estado]))
    
    return [], nos_expandidos

# Função para calcular a heurística (distância de Manhattan)
def heuristica(estado, estado_objetivo):
    distancia = 0
    for i in range(9):
        if estado[i] == 0:
            continue
        pos_obj = estado_objetivo.index(estado[i])
        pos_estado = i
        distancia += abs(pos_estado // 3 - pos_obj // 3) + abs(pos_estado % 3 - pos_obj % 3)
    return distancia

# Estados de exemplo para o puzzle de 8
estado_inicial = [1, 2, 3, 4, 5, 6, 7, 0, 8]  # Estado inicial
estado_objetivo = [0, 1, 2, 3, 4, 5, 6, 7, 8]  # Estado objetivo

# Pedir os algoritmos
algoritmos = {
    "procura em Extensão": procura_extensao,
    "procura em Profundidade": procura_profundidade,
    "procura de Custo Uniforme": procura_custo_uniforme,
    "procura sofrega": procura_sofrega,
    "procura A*": procura_a_star
}

for nome, algoritmo in algoritmos.items():
    if nome == "procura sofrega" or nome == "procura A*":
        caminho, nos_expandidos, tempo, num_nos_expandidos = medir_tempo_e_nos_expandidos(algoritmo, estado_inicial, estado_objetivo, heuristica)
    else:
        caminho, nos_expandidos, tempo, num_nos_expandidos = medir_tempo_e_nos_expandidos(algoritmo, estado_inicial, estado_objetivo)
    
    print(f"\n{nome}:")
    #print(f"Caminho: {caminho}")
    #print(f"Nos Expandidos: {nos_expandidos}")
    print(f"Numero de Nos Expandidos: {num_nos_expandidos}")
    print(f"Tempo de Execução: {tempo:.6f} segundos")

In [ ]:
# Procura com o Hill-Climbing
import copy
import random
import time
from typing import List, Tuple, Optional

class Puzzle8:
    """Classe que representa o puzzle-8"""
    
    def __init__(self, estado: List[List[int]] = None):
        """
        Inicializa o puzzle com um estado específico ou estado objetivo por padrão
        O estado é representado como uma matriz 3x3, onde 0 representa o espaço vazio
        """
        if estado:
            self.estado = estado
        else:
            # Estado objetivo padrão
            self.estado = [[1, 2, 3],
                          [4, 5, 6],
                          [7, 8, 0]]
        
        # Encontrar posição do espaço vazio (0)
        self.encontrar_vazio()
    
    def encontrar_vazio(self):
        """Encontra a posição do espaço vazio (0)"""
        for i in range(3):
            for j in range(3):
                if self.estado[i][j] == 0:
                    self.vazio_pos = (i, j)
                    return
    
    def mover(self, direcao: str) -> bool:
        """
        Move o espaço vazio na direção especificada
        Direções: 'cima', 'baixo', 'esquerda', 'direita'
        Retorna True se o movimento foi possível
        """
        i, j = self.vazio_pos
        novo_i, novo_j = i, j
        
        if direcao == 'cima' and i > 0:
            novo_i = i - 1
        elif direcao == 'baixo' and i < 2:
            novo_i = i + 1
        elif direcao == 'esquerda' and j > 0:
            novo_j = j - 1
        elif direcao == 'direita' and j < 2:
            novo_j = j + 1
        else:
            return False
        
        # Trocar posições
        self.estado[i][j], self.estado[novo_i][novo_j] = \
            self.estado[novo_i][novo_j], self.estado[i][j]
        
        # Atualizar posição do vazio
        self.vazio_pos = (novo_i, novo_j)
        return True
    
    def obter_vizinhos(self) -> List['Puzzle8']:
        """
        Gera todos os estados vizinhos possíveis
        Retorna uma lista de novos objetos Puzzle8
        """
        vizinhos = []
        direcoes = ['cima', 'baixo', 'esquerda', 'direita']
        
        for direcao in direcoes:
            novo_puzzle = copy.deepcopy(self)
            if novo_puzzle.mover(direcao):
                vizinhos.append(novo_puzzle)
        
        return vizinhos
    
    def calcular_heuristica_h1(self, objetivo: 'Puzzle8' = None) -> int:
        """
        Heurística H1: Número de peças mal colocadas
        Conta quantas peças estão em posições diferentes do estado objetivo
        """
        if objetivo is None:
            objetivo = Puzzle8()  # Estado objetivo padrão
        
        count = 0
        for i in range(3):
            for j in range(3):
                if self.estado[i][j] != 0 and self.estado[i][j] != objetivo.estado[i][j]:
                    count += 1
        return count
    
    def calcular_heuristica_h2(self, objetivo: 'Puzzle8' = None) -> int:
        """
        Heurística H2: Distância de Manhattan
        Soma das distâncias de cada peça até sua posição objetivo
        """
        if objetivo is None:
            objetivo = Puzzle8()
        
        # Mapear posições objetivo para cada número
        pos_objetivo = {}
        for i in range(3):
            for j in range(3):
                num = objetivo.estado[i][j]
                if num != 0:
                    pos_objetivo[num] = (i, j)
        
        # Calcular distância de Manhattan para cada peça
        distancia = 0
        for i in range(3):
            for j in range(3):
                num = self.estado[i][j]
                if num != 0:
                    i_objetivo, j_objetivo = pos_objetivo[num]
                    distancia += abs(i - i_objetivo) + abs(j - j_objetivo)
        
        return distancia
    
    def __eq__(self, other):
        """Compara dois puzzles para verificar se são iguais"""
        return self.estado == other.estado
    
    def __hash__(self):
        """Permite usar puzzle como chave em dicionários/sets"""
        return hash(str(self.estado))
    
    def __str__(self):
        """Representação em string do puzzle"""
        resultado = ""
        for linha in self.estado:
            resultado += " ".join(str(num) if num != 0 else " " for num in linha) + "\n"
        return resultado
    
    def __repr__(self):
        return f"Puzzle8({self.estado})"


class HillClimbingPuzzle8:
    """Implementação do algoritmo Hill-Climbing para o puzzle-8"""
    
    def __init__(self, heuristica='h2', max_iteracoes=1000):
        """
        Inicializa o solver Hill-Climbing
        heuristica: 'h1' para peças mal colocadas, 'h2' para distância de Manhattan
        max_iteracoes: número máximo de iterações para evitar loops infinitos
        """
        self.heuristica = heuristica
        self.max_iteracoes = max_iteracoes
        self.objetivo = Puzzle8()
        self.nos_visitados = 0
        self.historico_h = []  # Para análise da convergência
    
    def calcular_heuristica(self, puzzle: Puzzle8) -> int:
        """Calcula a heurística escolhida"""
        if self.heuristica == 'h1':
            return puzzle.calcular_heuristica_h1(self.objetivo)
        else:  # h2
            return puzzle.calcular_heuristica_h2(self.objetivo)
    
    def hill_climbing_basico(self, estado_inicial: Puzzle8) -> Tuple[Optional[Puzzle8], dict]:
        """
        Versão básica do Hill-Climbing
        Sempre escolhe o melhor vizinho (se for melhor que o atual)
        """
        atual = copy.deepcopy(estado_inicial)
        self.nos_visitados = 1
        self.historico_h = [self.calcular_heuristica(atual)]
        
        for iteracao in range(self.max_iteracoes):
            heuristica_atual = self.calcular_heuristica(atual)
            
            # Verificar se chegou ao objetivo
            if heuristica_atual == 0:
                return atual, {
                    'sucesso': True,
                    'iteracoes': iteracao + 1,
                    'nos_visitados': self.nos_visitados,
                    'historico_h': self.historico_h,
                    'tipo': 'básico'
                }
            
            # Gerar vizinhos e calcular suas heurísticas
            vizinhos = atual.obter_vizinhos()
            self.nos_visitados += len(vizinhos)
            
            if not vizinhos:
                break
            
            # Encontrar o melhor vizinho
            melhor_vizinho = None
            melhor_heuristica = float('inf')
            
            for vizinho in vizinhos:
                h_vizinho = self.calcular_heuristica(vizinho)
                if h_vizinho < melhor_heuristica:
                    melhor_heuristica = h_vizinho
                    melhor_vizinho = vizinho
            
            # Se o melhor vizinho não é melhor que o atual, parar (máximo local)
            if melhor_heuristica >= heuristica_atual:
                break
            
            # Mover para o melhor vizinho
            atual = melhor_vizinho
            self.historico_h.append(melhor_heuristica)
        
        return atual, {
            'sucesso': False,
            'iteracoes': iteracao + 1,
            'nos_visitados': self.nos_visitados,
            'historico_h': self.historico_h,
            'tipo': 'básico',
            'razao': 'máximo local atingido'
        }
    
    def hill_climbing_estocastico(self, estado_inicial: Puzzle8) -> Tuple[Optional[Puzzle8], dict]:
        """
        Versão estocástica do Hill-Climbing
        Escolhe aleatoriamente entre vizinhos melhores
        """
        atual = copy.deepcopy(estado_inicial)
        self.nos_visitados = 1
        self.historico_h = [self.calcular_heuristica(atual)]
        
        for iteracao in range(self.max_iteracoes):
            heuristica_atual = self.calcular_heuristica(atual)
            
            if heuristica_atual == 0:
                return atual, {
                    'sucesso': True,
                    'iteracoes': iteracao + 1,
                    'nos_visitados': self.nos_visitados,
                    'historico_h': self.historico_h,
                    'tipo': 'estocástico'
                }
            
            # Gerar vizinhos
            vizinhos = atual.obter_vizinhos()
            self.nos_visitados += len(vizinhos)
            
            # Filtrar vizinhos melhores
            vizinhos_melhores = []
            for vizinho in vizinhos:
                if self.calcular_heuristica(vizinho) < heuristica_atual:
                    vizinhos_melhores.append(vizinho)
            
            if not vizinhos_melhores:
                break  # Máximo local
            
            # Escolher aleatoriamente entre os melhores vizinhos
            atual = random.choice(vizinhos_melhores)
            self.historico_h.append(self.calcular_heuristica(atual))
        
        return atual, {
            'sucesso': False,
            'iteracoes': iteracao + 1,
            'nos_visitados': self.nos_visitados,
            'historico_h': self.historico_h,
            'tipo': 'estocástico',
            'razao': 'máximo local atingido'
        }
    
    def hill_climbing_com_random_restart(self, estado_inicial: Puzzle8, num_restarts=5) -> Tuple[Optional[Puzzle8], dict]:
        """
        Hill-Climbing com random restarts
        Tenta várias vezes a partir de estados aleatórios
        """
        melhor_solucao = None
        melhor_heuristica = float('inf')
        estatisticas = []
        
        for restart in range(num_restarts):
            if restart == 0:
                # Primeira tentativa com estado inicial
                estado_atual = copy.deepcopy(estado_inicial)
            else:
                # Gerar estado aleatório para restart
                estado_atual = self.gerar_estado_aleatorio()
            
            solucao, stats = self.hill_climbing_basico(estado_atual)
            stats['restart'] = restart + 1
            estatisticas.append(stats)
            
            h_solucao = self.calcular_heuristica(solucao)
            
            if h_solucao < melhor_heuristica:
                melhor_heuristica = h_solucao
                melhor_solucao = solucao
                
                if h_solucao == 0:
                    break
        
        return melhor_solucao, {
            'sucesso': melhor_heuristica == 0,
            'melhor_heuristica': melhor_heuristica,
            'num_restarts': restart + 1,
            'estatisticas': estatisticas,
            'tipo': 'random_restart'
        }
    
    def gerar_estado_aleatorio(self) -> Puzzle8:
        """Gera um estado aleatório do puzzle (solucionável)"""
        # Começar do objetivo e fazer movimentos aleatórios
        puzzle = Puzzle8()
        num_movimentos = random.randint(20, 50)
        
        for _ in range(num_movimentos):
            vizinhos = puzzle.obter_vizinhos()
            if vizinhos:
                puzzle = random.choice(vizinhos)
        
        return puzzle
    
    def resolver(self, estado_inicial: Puzzle8, metodo='basico', **kwargs) -> Tuple[Optional[Puzzle8], dict]:
        """
        Método principal para resolver o puzzle
        metodos: 'basico', 'estocastico', 'random_restart'
        """
        self.nos_visitados = 0
        self.historico_h = []
        
        if metodo == 'basico':
            return self.hill_climbing_basico(estado_inicial)
        elif metodo == 'estocastico':
            return self.hill_climbing_estocastico(estado_inicial)
        elif metodo == 'random_restart':
            num_restarts = kwargs.get('num_restarts', 5)
            return self.hill_climbing_com_random_restart(estado_inicial, num_restarts)
        else:
            raise ValueError(f"Método desconhecido: {metodo}")


def testar_algoritmos():
    """Função para testar os diferentes algoritmos"""
    
    # Estados de teste
    estados_teste = [
        # Estado objetivo (já resolvido)
        Puzzle8([[1, 2, 3],
                 [4, 5, 6],
                 [7, 8, 0]]),
        
        # Estado fácil (1 movimento)
        Puzzle8([[1, 2, 3],
                 [4, 5, 6],
                 [7, 0, 8]]),
        
        # Estado médio
        Puzzle8([[1, 2, 3],
                 [4, 0, 5],
                 [7, 8, 6]]),
        
        # Estado difícil
        Puzzle8([[2, 8, 3],
                 [1, 6, 4],
                 [7, 0, 5]]),
        
        # Estado muito difícil
        Puzzle8([[8, 1, 2],
                 [0, 4, 3],
                 [7, 6, 5]])
    ]
    
    print("=" * 80)
    print("TESTE DO ALGORITMO HILL-CLIMBING PARA PUZZLE-8")
    print("=" * 80)
    
    for i, estado in enumerate(estados_teste):
        print(f"\n\n--- Teste {i+1}: Estado Inicial ---")
        print(estado)
        
        # Testar diferentes heurísticas e métodos
        for heuristica in ['h1', 'h2']:
            print(f"\n  Heurística: {heuristica.upper()}")
            
            solver = HillClimbingPuzzle8(heuristica=heuristica, max_iteracoes=1000)
            
            for metodo in ['basico', 'estocastico']:
                print(f"\n    Método: {metodo.capitalize()}")
                
                solucao, stats = solver.resolver(estado, metodo=metodo)
                
                print(f"      Sucesso: {'sucesso' if stats['sucesso'] else 'falha'}")
                if not stats['sucesso'] and 'razao' in stats:
                    print(f"      Razão: {stats['razao']}")
                print(f"      Iterações: {stats['iteracoes']}")
                print(f"      Nós visitados: {stats['nos_visitados']}")
                
                if stats['sucesso']:
                    print(f"      Solução encontrada:")
                    print(solucao)
        
        # Testar random restart para estados difíceis
        if i >= 3:  # Para estados difíceis
            print(f"\n    Método: Random Restart (5 tentativas)")
            solver = HillClimbingPuzzle8(heuristica='h2')
            solucao, stats = solver.resolver(estado, metodo='random_restart', num_restarts=5)
            
            print(f"      Sucesso: {'sucesso' if stats['sucesso'] else 'falha'}")
            print(f"      Melhor heurística: {stats['melhor_heuristica']}")
            print(f"      Tentativas: {stats['num_restarts']}")


def jogar_interativo():
    """Função para o utilizador jogar ou testar um estado específico"""
    print("\n" + "=" * 60)
    print("MODO INTERATIVO - HILL-CLIMBING PARA PUZZLE-8")
    print("=" * 60)
    
    # Criar estado inicial personalizado
    print("\nIntroduza o estado inicial do puzzle (3x3, 0 para espaço vazio):")
    estado = []
    for i in range(3):
        linha = []
        for j in range(3):
            while True:
                try:
                    valor = int(input(f"  Posição [{i+1},{j+1}]: "))
                    if 0 <= valor <= 8:
                        linha.append(valor)
                        break
                    else:
                        print("    Valor deve estar entre 0 e 8!")
                except ValueError:
                    print("    Introduza um número válido!")
        estado.append(linha)
    
    puzzle_inicial = Puzzle8(estado)
    
    print("\nEstado inicial:")
    print(puzzle_inicial)
    
    # Escolher configuração
    print("\nConfiguração do algoritmo:")
    heuristica = input("Heurística (h1/h2) [padrão=h2]: ").strip() or 'h2'
    metodo = input("Método (basico/estocastico/random_restart) [padrão=basico]: ").strip() or 'basico'
    
    solver = HillClimbingPuzzle8(heuristica=heuristica)
    
    print("\nA resolver...")
    inicio = time.time()
    
    if metodo == 'random_restart':
        num_restarts = int(input("Número de restarts [padrão=5]: ").strip() or '5')
        solucao, stats = solver.resolver(puzzle_inicial, metodo=metodo, num_restarts=num_restarts)
    else:
        solucao, stats = solver.resolver(puzzle_inicial, metodo=metodo)
    
    fim = time.time()
    
    print(f"\nTempo de execução: {(fim - inicio)*1000:.2f} ms")
    print(f"Sucesso: {'sucesso' if stats['sucesso'] else 'falha'}")
    
    if metodo == 'random_restart':
        print(f"Melhor heurística: {stats['melhor_heuristica']}")
        print(f"Número de restarts: {stats['num_restarts']}")
    else:
        print(f"Iterações: {stats['iteracoes']}")
        print(f"Nós visitados: {stats['nos_visitados']}")
    
    if 'historico_h' in stats and stats['historico_h']:
        print(f"Evolução da heurística: {stats['historico_h']}")
    
    if stats['sucesso']:
        print("\nSolução encontrada:")
        print(solucao)
    else:
        print("\nMelhor estado encontrado:")
        print(solucao)


print("Escolha uma opção:")
print("1 - Executar testes automáticos")
print("2 - Modo interativo")
    
opcao = input("Opção [1]: ").strip() or '1'
    
if opcao == '1':
    testar_algoritmos()
else:
    jogar_interativo()


# Exemplo de uso básico
#puzzle = Puzzle8([[2, 8, 3], [1, 6, 4], [7, 0, 5]])
#solver = HillClimbingPuzzle8(heuristica='h2')
#solucao, stats = solver.resolver(puzzle, metodo='estocastico')

#if stats['sucesso']:
#    print("Solução encontrada!")
#    print(solucao)
#else:
#    print(f"Falhou: {stats['razao']}")


In [ ]:
# Procura com o Simulated Anneling
import copy
import random
import math
import time
from typing import List, Tuple, Optional
import matplotlib.pyplot as plt

class Puzzle8:
    """Classe que representa o puzzle-8"""
    
    def __init__(self, estado: List[List[int]] = None):
        """
        Inicializa o puzzle com um estado específico ou estado objetivo por padrão
        O estado é representado como uma matriz 3x3, onde 0 representa o espaço vazio
        """
        if estado:
            self.estado = estado
        else:
            # Estado objetivo padrão
            self.estado = [[1, 2, 3],
                          [4, 5, 6],
                          [7, 8, 0]]
        
        # Encontrar posição do espaço vazio (0)
        self.encontrar_vazio()
    
    def encontrar_vazio(self):
        """Encontra a posição do espaço vazio (0)"""
        for i in range(3):
            for j in range(3):
                if self.estado[i][j] == 0:
                    self.vazio_pos = (i, j)
                    return
    
    def mover(self, direcao: str) -> bool:
        """
        Move o espaço vazio na direção especificada
        Direções: 'cima', 'baixo', 'esquerda', 'direita'
        Retorna True se o movimento foi possível
        """
        i, j = self.vazio_pos
        novo_i, novo_j = i, j
        
        if direcao == 'cima' and i > 0:
            novo_i = i - 1
        elif direcao == 'baixo' and i < 2:
            novo_i = i + 1
        elif direcao == 'esquerda' and j > 0:
            novo_j = j - 1
        elif direcao == 'direita' and j < 2:
            novo_j = j + 1
        else:
            return False
        
        # Trocar posições
        self.estado[i][j], self.estado[novo_i][novo_j] = \
            self.estado[novo_i][novo_j], self.estado[i][j]
        
        # Atualizar posição do vazio
        self.vazio_pos = (novo_i, novo_j)
        return True
    
    def obter_vizinhos(self) -> List['Puzzle8']:
        """
        Gera todos os estados vizinhos possíveis
        Retorna uma lista de novos objetos Puzzle8
        """
        vizinhos = []
        direcoes = ['cima', 'baixo', 'esquerda', 'direita']
        
        for direcao in direcoes:
            novo_puzzle = copy.deepcopy(self)
            if novo_puzzle.mover(direcao):
                vizinhos.append(novo_puzzle)
        
        return vizinhos
    
    def obter_vizinho_aleatorio(self) -> Optional['Puzzle8']:
        """
        Retorna um vizinho aleatório
        Útil para o Simulated Annealing
        """
        vizinhos = self.obter_vizinhos()
        if vizinhos:
            return random.choice(vizinhos)
        return None
    
    def calcular_heuristica_h1(self, objetivo: 'Puzzle8' = None) -> int:
        """
        Heurística H1: Número de peças mal colocadas
        Conta quantas peças estão em posições diferentes do estado objetivo
        """
        if objetivo is None:
            objetivo = Puzzle8()  # Estado objetivo padrão
        
        count = 0
        for i in range(3):
            for j in range(3):
                if self.estado[i][j] != 0 and self.estado[i][j] != objetivo.estado[i][j]:
                    count += 1
        return count
    
    def calcular_heuristica_h2(self, objetivo: 'Puzzle8' = None) -> int:
        """
        Heurística H2: Distância de Manhattan
        Soma das distâncias de cada peça até sua posição objetivo
        """
        if objetivo is None:
            objetivo = Puzzle8()
        
        # Mapear posições objetivo para cada número
        pos_objetivo = {}
        for i in range(3):
            for j in range(3):
                num = objetivo.estado[i][j]
                if num != 0:
                    pos_objetivo[num] = (i, j)
        
        # Calcular distância de Manhattan para cada peça
        distancia = 0
        for i in range(3):
            for j in range(3):
                num = self.estado[i][j]
                if num != 0:
                    i_objetivo, j_objetivo = pos_objetivo[num]
                    distancia += abs(i - i_objetivo) + abs(j - j_objetivo)
        
        return distancia
    
    def __eq__(self, other):
        """Compara dois puzzles para verificar se são iguais"""
        return self.estado == other.estado
    
    def __hash__(self):
        """Permite usar puzzle como chave em dicionários/sets"""
        return hash(str(self.estado))
    
    def __str__(self):
        """Representação em string do puzzle"""
        resultado = ""
        for linha in self.estado:
            resultado += " ".join(str(num) if num != 0 else " " for num in linha) + "\n"
        return resultado


class SimulatedAnnealingPuzzle8:
    """
    Implementação do algoritmo Simulated Annealing para o puzzle-8
    Inspirado no processo de recozimento de metais
    """
    
    def __init__(self, heuristica='h2', 
                 temperatura_inicial=100.0,
                 temperatura_final=0.01,
                 alpha=0.95,
                 max_iteracoes_por_temperatura=100):
        """
        Inicializa o solver Simulated Annealing
        
        Parâmetros:
        - heuristica: 'h1' ou 'h2'
        - temperatura_inicial: temperatura inicial (deve ser alta)
        - temperatura_final: temperatura onde o algoritmo para
        - alpha: fator de arrefecimento (0 < alpha < 1)
        - max_iteracoes_por_temperatura: iterações em cada temperatura
        """
        self.heuristica = heuristica
        self.temperatura_inicial = temperatura_inicial
        self.temperatura_final = temperatura_final
        self.alpha = alpha
        self.max_iteracoes_por_temperatura = max_iteracoes_por_temperatura
        
        self.objetivo = Puzzle8()
        self.nos_visitados = 0
        self.historico_temperatura = []
        self.historico_energia = []
        self.historico_aceitacao = []
    
    def calcular_energia(self, puzzle: Puzzle8) -> int:
        """
        Calcula a energia (custo) do estado
        Quanto menor a energia, melhor o estado
        """
        if self.heuristica == 'h1':
            return puzzle.calcular_heuristica_h1(self.objetivo)
        else:  # h2
            return puzzle.calcular_heuristica_h2(self.objetivo)
    
    def probabilidade_aceitacao(self, delta_energia: float, temperatura: float) -> float:
        """
        Calcula a probabilidade de aceitar um movimento para pior estado
        Usa a distribuição de Boltzmann: P = exp(-ΔE / T)
        """
        if temperatura <= 0:
            return 0
        if delta_energia < 0:
            return 1.0  # Sempre aceita melhorias
        return math.exp(-delta_energia / temperatura)
    
    def schedule_temperatura(self, temperatura_atual: float, iteracao: int) -> float:
        """
        Agenda de resfriamento geométrico
        Também conhecido como exponential cooling
        """
        return temperatura_atual * self.alpha
    
    def simulated_annealing(self, estado_inicial: Puzzle8, 
                           verbose: bool = False) -> Tuple[Optional[Puzzle8], dict]:
        """
        Versão básica do Simulated Annealing
        """
        atual = copy.deepcopy(estado_inicial)
        melhor = copy.deepcopy(atual)
        
        energia_atual = self.calcular_energia(atual)
        melhor_energia = energia_atual
        
        temperatura = self.temperatura_inicial
        iteracao = 0
        self.nos_visitados = 1
        self.historico_temperatura = [temperatura]
        self.historico_energia = [energia_atual]
        self.historico_aceitacao = []
        
        total_iteracoes = 0
        aceitacoes = 0
        
        while temperatura > self.temperatura_final:
            for _ in range(self.max_iteracoes_por_temperatura):
                # Gerar vizinho aleatório
                vizinho = atual.obter_vizinho_aleatorio()
                if vizinho is None:
                    continue
                
                self.nos_visitados += 1
                energia_vizinho = self.calcular_energia(vizinho)
                delta_energia = energia_vizinho - energia_atual
                
                # Calcular probabilidade de aceitação
                prob = self.probabilidade_aceitacao(delta_energia, temperatura)
                
                # Decidir se aceita o movimento
                if random.random() < prob:
                    atual = vizinho
                    energia_atual = energia_vizinho
                    aceitacoes += 1
                    
                    # Atualizar melhor solução
                    if energia_atual < melhor_energia:
                        melhor = copy.deepcopy(atual)
                        melhor_energia = energia_atual
                
                total_iteracoes += 1
                
                # Verificar se encontrou solução ótima
                if melhor_energia == 0:
                    break
            
            # Registar histórico
            self.historico_temperatura.append(temperatura)
            self.historico_energia.append(melhor_energia)
            self.historico_aceitacao.append(aceitacoes / max(1, total_iteracoes))
            
            if verbose:
                print(f"T={temperatura:.2f}, Melhor energia={melhor_energia}, "
                      f"Aceitações={aceitacoes}/{total_iteracoes}")
            
            # Resfriar
            temperatura = self.schedule_temperatura(temperatura, iteracao)
            iteracao += 1
            
            if melhor_energia == 0:
                break
        
        return melhor, {
            'sucesso': melhor_energia == 0,
            'energia_final': melhor_energia,
            'iteracoes': iteracao,
            'iteracoes_totais': total_iteracoes,
            'nos_visitados': self.nos_visitados,
            'temperatura_final': temperatura,
            'taxa_aceitacao': aceitacoes / max(1, total_iteracoes),
            'historico_temperatura': self.historico_temperatura,
            'historico_energia': self.historico_energia,
            'historico_aceitacao': self.historico_aceitacao
        }
    
    def simulated_annealing_com_reativação(self, estado_inicial: Puzzle8,
                                          num_reativacoes: int = 3,
                                          verbose: bool = False) -> Tuple[Optional[Puzzle8], dict]:
        """
        Simulated Annealing com reativação de temperatura
        Útil para escapar de mínimos locais
        """
        melhor_global = None
        melhor_energia_global = float('inf')
        estatisticas = []
        
        temperatura_original = self.temperatura_inicial
        
        for reativacao in range(num_reativacoes):
            if verbose:
                print(f"\nReativação {reativacao + 1}/{num_reativacoes}")
            
            # Reiniciar temperatura
            self.temperatura_inicial = temperatura_original / (reativacao + 1)
            
            # Se não é a primeira reativação, usar melhor encontrado como inicial
            if reativacao == 0:
                inicial = estado_inicial
            else:
                inicial = melhor_global if melhor_global else estado_inicial
            
            solucao, stats = self.simulated_annealing(inicial, verbose=False)
            stats['reativacao'] = reativacao + 1
            estatisticas.append(stats)
            
            energia_atual = self.calcular_energia(solucao)
            if energia_atual < melhor_energia_global:
                melhor_energia_global = energia_atual
                melhor_global = solucao
                
                if energia_atual == 0:
                    break
        
        # Restaurar temperatura original
        self.temperatura_inicial = temperatura_original
        
        return melhor_global, {
            'sucesso': melhor_energia_global == 0,
            'melhor_energia': melhor_energia_global,
            'num_reativacoes': reativacao + 1,
            'estatisticas': estatisticas,
            'tipo': 'com_reativacao'
        }
    
    def resolver(self, estado_inicial: Puzzle8, 
                metodo='basico',
                verbose=False,
                **kwargs) -> Tuple[Optional[Puzzle8], dict]:
        """
        Método principal para resolver o puzzle
        
        Métodos:
        - 'basico': Simulated Annealing padrão
        - 'reativacao': Com reativação de temperatura
        """
        self.nos_visitados = 0
        self.historico_temperatura = []
        self.historico_energia = []
        
        if metodo == 'basico':
            return self.simulated_annealing(estado_inicial, verbose)
        elif metodo == 'reativacao':
            num_reativacoes = kwargs.get('num_reativacoes', 3)
            return self.simulated_annealing_com_reativação(
                estado_inicial, num_reativacoes, verbose)
        else:
            raise ValueError(f"Método desconhecido: {metodo}")


class ComparadorAlgoritmos:
    """Classe para comparar diferentes configurações de Simulated Annealing"""
    
    def __init__(self):
        self.resultados = []
    
    def testar_configuracoes(self, estado_inicial: Puzzle8, num_testes: int = 5):
        """
        Testa diferentes configurações de parâmetros
        """
        configuracoes = [
            {'nome': 'Rápido (Alta taxa de arrefecimento)',
             'temp_inicial': 100, 'alpha': 0.8, 'iter_por_temp': 50},
            
            {'nome': 'Padrão',
             'temp_inicial': 100, 'alpha': 0.95, 'iter_por_temp': 100},
            
            {'nome': 'Lento (Baixa taxa de resfriamento)',
             'temp_inicial': 100, 'alpha': 0.99, 'iter_por_temp': 200},
            
            {'nome': 'Alta temperatura inicial',
             'temp_inicial': 500, 'alpha': 0.95, 'iter_por_temp': 100},
            
            {'nome': 'Muitas iterações por temperatura',
             'temp_inicial': 100, 'alpha': 0.95, 'iter_por_temp': 500}
        ]
        
        print("\n" + "=" * 80)
        print("COMPARAÇÃO DE CONFIGURAÇÕES DE SIMULATED ANNEALING")
        print("=" * 80)
        
        for config in configuracoes:
            print(f"\n--- {config['nome']} ---")
            sucessos = 0
            energias = []
            tempos = []
            
            for teste in range(num_testes):
                solver = SimulatedAnnealingPuzzle8(
                    heuristica='h2',
                    temperatura_inicial=config['temp_inicial'],
                    alpha=config['alpha'],
                    max_iteracoes_por_temperatura=config['iter_por_temp']
                )
                
                inicio = time.time()
                solucao, stats = solver.resolver(estado_inicial, metodo='basico')
                fim = time.time()
                
                tempo = fim - inicio
                energia = stats['energia_final']
                
                if stats['sucesso']:
                    sucessos += 1
                
                energias.append(energia)
                tempos.append(tempo)
            
            print(f"  Taxa de sucesso: {sucessos}/{num_testes} ({sucessos/num_testes*100:.1f}%)")
            print(f"  Energia média: {sum(energias)/len(energias):.2f}")
            print(f"  Melhor energia: {min(energias)}")
            print(f"  Tempo médio: {sum(tempos)/len(tempos)*1000:.2f} ms")


def visualizar_convergencia(historico_energia: List[float], 
                           historico_temperatura: List[float],
                           titulo: str = "Convergência do Simulated Annealing"):
    """
    Visualiza a convergência do algoritmo
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
    
    # Gráfico de energia
    ax1.plot(historico_energia, 'b-', linewidth=2)
    ax1.set_xlabel('Iteração')
    ax1.set_ylabel('Energia (Heurística)')
    ax1.set_title('Evolução da Energia')
    ax1.grid(True, alpha=0.3)
    
    # Gráfico de temperatura
    ax2.plot(historico_temperatura, 'r-', linewidth=2)
    ax2.set_xlabel('Iteração')
    ax2.set_ylabel('Temperatura')
    ax2.set_title('Agenda de Arrefecimento')
    ax2.grid(True, alpha=0.3)
    ax2.set_yscale('log')  # Escala logarítmica para melhor visualização
    
    plt.suptitle(titulo)
    plt.tight_layout()
    plt.show()


def jogar_interativo():
    """Função para o utilizador testar o Simulated Annealing"""
    print("\n" + "=" * 60)
    print("SIMULATED ANNEALING PARA PUZZLE-8 - MODO INTERATIVO")
    print("=" * 60)
    
    # Criar estado inicial personalizado
    print("\nIntroduza o estado inicial do puzzle (3x3, 0 para espaço vazio):")
    estado = []
    for i in range(3):
        linha = []
        for j in range(3):
            while True:
                try:
                    valor = int(input(f"  Posição [{i+1},{j+1}]: "))
                    if 0 <= valor <= 8:
                        linha.append(valor)
                        break
                    else:
                        print("    Valor deve estar entre 0 e 8!")
                except ValueError:
                    print("    Introduza um número válido!")
        estado.append(linha)
    
    puzzle_inicial = Puzzle8(estado)
    
    print("\nEstado inicial:")
    print(puzzle_inicial)
    
    # Escolher configuração
    print("\nConfiguração do Simulated Annealing:")
    
    heuristica = input("Heurística (h1/h2) [padrão=h2]: ").strip() or 'h2'
    temp_inicial = float(input("Temperatura inicial [padrão=100]: ").strip() or '100')
    temp_final = float(input("Temperatura final [padrão=0.01]: ").strip() or '0.01')
    alpha = float(input("Fator de arrefecimento (0.8-0.99) [padrão=0.95]: ").strip() or '0.95')
    iter_por_temp = int(input("Iterações por temperatura [padrão=100]: ").strip() or '100')
    
    metodo = input("Método (basico/reativacao) [padrão=basico]: ").strip() or 'basico'
    
    solver = SimulatedAnnealingPuzzle8(
        heuristica=heuristica,
        temperatura_inicial=temp_inicial,
        temperatura_final=temp_final,
        alpha=alpha,
        max_iteracoes_por_temperatura=iter_por_temp
    )
    
    print("\nA resolver...")
    inicio = time.time()
    
    if metodo == 'reativacao':
        num_reativacoes = int(input("Número de reativações [padrão=3]: ").strip() or '3')
        solucao, stats = solver.resolver(puzzle_inicial, 
                                        metodo=metodo,
                                        num_reativacoes=num_reativacoes,
                                        verbose=True)
    else:
        solucao, stats = solver.resolver(puzzle_inicial, 
                                        metodo=metodo,
                                        verbose=True)
    
    fim = time.time()
    
    print(f"\n" + "=" * 50)
    print("RESULTADOS")
    print("=" * 50)
    print(f"Tempo de execução: {(fim - inicio)*1000:.2f} ms")
    print(f"Sucesso: {'Sucesso' if stats['sucesso'] else 'Falha'}")
    print(f"Energia final: {stats['energia_final']}")
    print(f"Iterações: {stats['iteracoes']}")
    print(f"Iterações totais: {stats['iteracoes_totais']}")
    print(f"Nós visitados: {stats['nos_visitados']}")
    print(f"Taxa de aceitação: {stats['taxa_aceitacao']*100:.1f}%")
    
    if metodo == 'reativacao':
        print(f"Número de reativações: {stats['num_reativacoes']}")
    
    if stats['sucesso']:
        print("\nSolução encontrada:")
        print(solucao)
    else:
        print("\nMelhor estado encontrado:")
        print(solucao)
    
    # Perguntar se quer visualizar a convergência
    visualizar = input("\nVisualizar gráfico de convergência? (s/n): ").strip().lower()
    if visualizar == 's':
        visualizar_convergencia(
            stats['historico_energia'],
            stats['historico_temperatura'],
            f"Convergência - {metodo.capitalize()}"
        )


def exemplo_detalhado():
    """Exemplo detalhado do funcionamento do Simulated Annealing"""
    
    # Estado difícil
    estado_dificil = Puzzle8([[8, 1, 2],
                              [0, 4, 3],
                              [7, 6, 5]])
    
    print("=" * 80)
    print("EXEMPLO DETALHADO - SIMULATED ANNEALING")
    print("=" * 80)
    print("\nEstado inicial:")
    print(estado_dificil)
    
    # Configurar solver
    solver = SimulatedAnnealingPuzzle8(
        heuristica='h2',
        temperatura_inicial=200,  # Temperatura alta para explorar
        temperatura_final=0.1,
        alpha=0.95,
        max_iteracoes_por_temperatura=100
    )
    
    print("\nA resolver com Simulated Annealing básico...")
    solucao, stats = solver.resolver(estado_dificil, metodo='basico', verbose=True)
    
    print("\n" + "=" * 50)
    print("RESULTADO FINAL")
    print("=" * 50)
    print(f"Sucesso: {'Sucesso' if stats['sucesso'] else 'Falha'}")
    print(f"Energia final: {stats['energia_final']}")
    print(f"Iterações: {stats['iteracoes']}")
    print(f"Nós visitados: {stats['nos_visitados']}")
    print(f"Taxa de aceitação: {stats['taxa_aceitacao']*100:.1f}%")
    
    if stats['sucesso']:
        print("\nSolução encontrada:")
        print(solucao)
    
    # Visualizar convergência
    visualizar_convergencia(stats['historico_energia'], stats['historico_temperatura'])


print("\nEscolha uma opção:")
print("1 - Exemplo detalhado")
print("2 - Modo interativo")
print("3 - Comparar configurações")
    
opcao = input("Opção [1]: ").strip() or '1'
    
if opcao == '1':
    exemplo_detalhado()
elif opcao == '2':
    jogar_interativo()
elif opcao == '3':
    # Estado para teste
    estado_teste = Puzzle8([[2, 8, 3],
                            [1, 6, 4],
                            [7, 0, 5]])
    comparador = ComparadorAlgoritmos()
    comparador.testar_configuracoes(estado_teste, num_testes=3)

# Exemplo de utilização básico
#puzzle = Puzzle8([[2, 8, 3], [1, 6, 4], [7, 0, 5]])
#solver = SimulatedAnnealingPuzzle8(
#    temperatura_inicial=100,
#    alpha=0.95,
#    max_iteracoes_por_temperatura=100
#)
#solucao, stats = solver.resolver(puzzle, metodo='basico')

#if stats['sucesso']:
#    print("Solução encontrada!")
#    print(solucao)

In [ ]:
"""
Implementação do Algoritmo Genético - passo a passo
"""

import numpy as np
import random

def criar_solucao_referencia(tamanho):

    numero_de_uns = int(tamanho / 2)

    # Construir matriz com mistura igual de zero e uns
    referencia = np.zeros(tamanho)
    referencia[0: numero_de_uns] = 1

    # Misturar na matriz zeros e uns
    np.random.shuffle(referencia)
    
    return referencia

# Mostrar população de referência
print(criar_solucao_referencia(70))

In [ ]:
# Criar população
def criar_populacao_inicial(individuos, tamanho):
    # Set up an initial array of all zeros
    populacao = np.zeros((individuos, tamanho))
    # Ciclo para cada linha dos individuos
    for i in range(individuos):
        # Escolher um numero aleatório de 1s para criar
        uns = random.randint(0, tamanho)
        populacao[i, 0:uns] = 1
        # Misturar
        np.random.shuffle(populacao[i])
    
    return populacao

print (criar_populacao_inicial(4, 10))

In [ ]:
# Mostrar pontuação ou fitness da população
def calcular_fitness(referencia, populacao):
    # Criar um array V/F comparado com a referencia
    identico_referencia = populacao == referencia
    # Soma o numero de genes idêntico ao da referencia
    fitness_score = identico_referencia.sum(axis=1)
    
    return fitness_score

referencia = criar_solucao_referencia(10)
print ('Solucao Referencia: \n', referencia)
populacao = criar_populacao_inicial(6, 10)
print ('\nPopulacao inicial: \n', populacao)
score = calcular_fitness(referencia, populacao)
print('\nPontuacao: \n', score)

In [ ]:
# Seleccionar dois pais e mostrar

def selecionar_individuos_por_torneio(populacao, score):
    # Obter o tamanho da populacao
    tamanho_populacao = len(score)
    
    # Escolher individuos para torneio
    lutador_1 = random.randint(0, tamanho_populacao-1)
    lutador_2 = random.randint(0, tamanho_populacao-1)
    
    # Obter o fitness de cada
    lutador_1_fitness = score[lutador_1]
    lutador_2_fitness = score[lutador_2]
    
    # Identifica o individuo com fitness mais alto
    # O lutador 1 vai vencer se o score for igual
    if lutador_1_fitness >= lutador_2_fitness:
        vencedor = lutador_1
    else:
        vencedor = lutador_2
    
    # Retorna o cromosoma do vencedor
    return populacao[vencedor, :]

pai_1 = selecionar_individuos_por_torneio(populacao, score)
pai_2 = selecionar_individuos_por_torneio(populacao, score)
    
print("Pai1 :", pai_1)
print("Pai2 :", pai_2)

In [ ]:
# Obter filhos
def produzir_por_crossover(pai_1, pai_2):
    # Obter o tamanho do cromosoma
    tamanho = len(pai_1)
    
    # Escolher o ponto de cruzamento, evitando fins do cromosoma
    ponto = random.randint(1,tamanho-1)
    
    # Criar filhos. np.hstack junta dois arrays
    filho_1 = np.hstack((pai_1[0: ponto],
                        pai_2[ponto:]))
    
    filho_2 = np.hstack((pai_2[0: ponto],
                        pai_1[ponto:]))
    
    # Retornar filhos
    return filho_1, filho_2
    
filho_1, filho_2 = produzir_por_crossover(pai_1, pai_2)

print("Filho1 :", filho_1)
print("Filho2 :", filho_2)

In [ ]:
# operador Mutação
def mutacao_aleatoria_populacao(populacao, probabilidade_mutacao):
    
    # Aplicar mutacao aleatoria
    array = np.random.random(size=(populacao.shape))
    
    bam = \
        array <= probabilidade_mutacao

    populacao[bam] = \
        np.logical_not(populacao[bam])
        
    # Retorna a população de mutação
    return populacao

populacao = np.stack((filho_1, filho_2))
# Mutar a população
probabilidade_mutacao = 0.25
print("População antes da mutação:")
print(populacao)
    
populacao = mutacao_aleatoria_populacao(populacao, probabilidade_mutacao)
print("População depois da mutação:")
print(populacao)

In [ ]:
"""
Exemplo Simples de Algoritmo Genético

Para a função x^2
"""

import random

# Parâmetros do algoritmo
tamanho_populacao = 6
taxa_mutacao = 0.1
geracoes = 10

# Função objetivo (maximizar f(x) = x^2)
def funcao_objetivo(x):
    return x ** 2

# Gerar população inicial (números binários de 5 bits)
def criar_populacao(tamanho):
    return [format(random.randint(0, 31), '05b') for _ in range(tamanho)]

# Avaliar a aptidão de cada indivíduo
def avaliar_populacao(populacao):
    return [funcao_objetivo(int(individuo, 2)) for individuo in populacao]

# Seleção por torneio
def selecao_torneio(populacao, aptidoes):
    selecionados = []
    for _ in range(len(populacao)):
        i1, i2 = random.sample(range(len(populacao)), 2)
        melhor = populacao[i1] if aptidoes[i1] > aptidoes[i2] else populacao[i2]
        selecionados.append(melhor)
    return selecionados

# Crossover de um ponto
def crossover(pai1, pai2):
    ponto = random.randint(1, 4)
    filho1 = pai1[:ponto] + pai2[ponto:]
    filho2 = pai2[:ponto] + pai1[ponto:]
    return filho1, filho2

# Mutação aleatória
def mutacao(individuo):
    if random.random() < taxa_mutacao:
        pos = random.randint(0, 4)
        individuo = individuo[:pos] + ('1' if individuo[pos] == '0' else '0') + individuo[pos+1:]
    return individuo

# Algoritmo Genético principal
def algoritmo_genetico():
    populacao = criar_populacao(tamanho_populacao)
    
    for geracao in range(geracoes):
        aptidoes = avaliar_populacao(populacao)
        print(f'Geração {geracao}: Melhor solução -> {max(aptidoes)}')
        
        populacao = selecao_torneio(populacao, aptidoes)
        nova_populacao = []
        
        for i in range(0, tamanho_populacao, 2):
            filho1, filho2 = crossover(populacao[i], populacao[i+1])
            nova_populacao.extend([mutacao(filho1), mutacao(filho2)])
        
        populacao = nova_populacao
    
    melhor_x = max(populacao, key=lambda ind: funcao_objetivo(int(ind, 2)))
    print(f'Melhor solução final: x = {int(melhor_x, 2)}, f(x) = {funcao_objetivo(int(melhor_x, 2))}')

# Rodar o algoritmo
algoritmo_genetico()

In [ ]:
"""
Algoritmo Genético para resolver o problema do mapa da Roménia
"""

import random
import numpy as np
from operator import itemgetter

# Mapa da Romênia corrigido e completo
romania_map = {
    'Arad': {'Zerind': 75, 'Sibiu': 140, 'Timisoara': 118},
    'Zerind': {'Arad': 75, 'Oradea': 71},
    'Oradea': {'Zerind': 71, 'Sibiu': 151},
    'Sibiu': {'Arad': 140, 'Oradea': 151, 'Fagaras': 99, 'Rimnicu Vilcea': 80},
    'Timisoara': {'Arad': 118, 'Lugoj': 111},
    'Lugoj': {'Timisoara': 111, 'Mehadia': 70},
    'Mehadia': {'Lugoj': 70, 'Drobeta': 75},
    'Drobeta': {'Mehadia': 75, 'Craiova': 120},
    'Craiova': {'Drobeta': 120, 'Rimnicu Vilcea': 146, 'Pitesti': 138},
    'Rimnicu Vilcea': {'Sibiu': 80, 'Craiova': 146, 'Pitesti': 97},
    'Fagaras': {'Sibiu': 99, 'Bucharest': 211},
    'Pitesti': {'Rimnicu Vilcea': 97, 'Craiova': 138, 'Bucharest': 101},
    'Bucharest': {'Fagaras': 211, 'Pitesti': 101, 'Giurgiu': 90, 'Urziceni': 85},
    'Giurgiu': {'Bucharest': 90},
    'Urziceni': {'Bucharest': 85, 'Hirsova': 98, 'Vaslui': 142},
    'Hirsova': {'Urziceni': 98, 'Eforie': 86},
    'Eforie': {'Hirsova': 86},
    'Vaslui': {'Urziceni': 142, 'Iasi': 92},
    'Iasi': {'Vaslui': 92, 'Neamt': 87},
    'Neamt': {'Iasi': 87}
}

# Garantir que todas as conexões são bidirecionais
for city in list(romania_map.keys()):
    for neighbor, dist in romania_map[city].items():
        if city not in romania_map[neighbor]:
            romania_map[neighbor][city] = dist

# Parâmetros do algoritmo genético
POPULATION_SIZE = 100
GENERATIONS = 200
MUTATION_RATE = 0.15
ELITISM_RATE = 0.1
CROSSOVER_RATE = 0.85

cities = list(romania_map.keys())

def create_individual(start_city, end_city, max_length=15):
    """Cria um caminho válido começando em start_city"""
    path = [start_city]
    current = start_city
    
    while current != end_city and len(path) < max_length:
        neighbors = list(romania_map[current].keys())
        next_city = random.choice(neighbors)
        path.append(next_city)
        current = next_city
    
    return path

def calculate_fitness(path, end_city):
    """Calcula a distância total do caminho"""
    try:
        if path[-1] != end_city:
            return float('inf')  # Penaliza caminhos que não chegam ao destino
        
        total = 0
        for i in range(len(path)-1):
            if path[i+1] not in romania_map[path[i]]:
                return float('inf')  # Conexão inválida
            total += romania_map[path[i]][path[i+1]]
        
        return total
    except:
        return float('inf')  # Para qualquer outro erro

def crossover(p1, p2):
    """Cruzamento de caminhos"""
    if random.random() > CROSSOVER_RATE or len(p1) < 2 or len(p2) < 2:
        return p1.copy(), p2.copy()
    
    # Encontra cidades em comum
    common = set(p1) & set(p2)
    if not common:
        return p1.copy(), p2.copy()
    
    crossover_point = random.choice(list(common))
    idx1 = p1.index(crossover_point)
    idx2 = p2.index(crossover_point)
    
    child1 = p1[:idx1] + p2[idx2:]
    child2 = p2[:idx2] + p1[idx1:]
    
    return child1, child2

def mutate(path, end_city):
    """Aplica mutações ao caminho"""
    if random.random() > MUTATION_RATE or len(path) < 2:
        return path
    
    new_path = path.copy()
    mutation_type = random.choice(['add', 'remove', 'replace'])
    
    try:
        if mutation_type == 'add' and len(new_path) < 15:
            pos = random.randint(1, len(new_path)-1)
            current = new_path[pos-1]
            neighbors = list(romania_map[current].keys())
            new_city = random.choice(neighbors)
            new_path.insert(pos, new_city)
            
        elif mutation_type == 'remove' and len(new_path) > 2:
            pos = random.randint(1, len(new_path)-1)
            new_path.pop(pos)
            
        elif mutation_type == 'replace' and len(new_path) > 1:
            pos = random.randint(1, len(new_path)-1)
            current = new_path[pos-1]
            neighbors = list(romania_map[current].keys())
            new_city = random.choice(neighbors)
            new_path[pos] = new_city
        
        # Garante que termina no destino
        if new_path[-1] != end_city and len(new_path) < 15:
            current = new_path[-1]
            neighbors = list(romania_map[current].keys())
            if end_city in neighbors:
                new_path.append(end_city)
            else:
                new_path.append(random.choice(neighbors))
    
    except:
        return path  # Se der erro, retorna o original
    
    return new_path

def genetic_algorithm(start, end):
    """Executa o algoritmo genético"""
    population = [create_individual(start, end) for _ in range(POPULATION_SIZE)]
    best_dist = float('inf')
    best_path = None
    history = []
    
    for generation in range(GENERATIONS):
        # Avaliação
        fitness = [calculate_fitness(ind, end) for ind in population]
        current_best = min(fitness)
        history.append(current_best)
        
        if current_best < best_dist:
            best_dist = current_best
            best_path = population[fitness.index(current_best)]
        
        # Elitismo
        elite_size = int(ELITISM_RATE * POPULATION_SIZE)
        elite_indices = np.argsort(fitness)[:elite_size]
        elite = [population[i] for i in elite_indices]
        
        # Seleção por torneio
        parents = []
        for _ in range(POPULATION_SIZE - elite_size):
            candidates = random.sample(list(zip(population, fitness)), 3)
            winner = min(candidates, key=itemgetter(1))[0]
            parents.append(winner)
        
        # Cruzamento e mutação
        offspring = []
        for i in range(0, len(parents), 2):
            if i+1 >= len(parents):
                offspring.append(parents[i])
                continue
                
            child1, child2 = crossover(parents[i], parents[i+1])
            offspring.append(mutate(child1, end))
            offspring.append(mutate(child2, end))
        
        population = elite + offspring
    
    return best_path, best_dist, history

# Execução
start_city = 'Arad'
end_city = 'Bucharest'
solution, distance, fitness_history = genetic_algorithm(start_city, end_city)

print(f"\nMelhor caminho de {start_city} para {end_city}:")
print(" -> ".join(solution))
print(f"Distância total: {distance} km")

In [ ]:
"""
Algoritmo genético - Puzzle8
"""
import numpy as np
import math
import random
from copy import deepcopy
from random import randint, choice

class Tabuleiro:
    def __init__(self):
        self.board = np.zeros((3, 3))

        index = 0
        for i in range (3):
            for j in range (3):
                self.board[i, j] = index
                index += 1

    def swap(self, a, b):
        temp = self.board[a]
        self.board[a] = self.board[b]
        self.board[b] = temp

    def move_left(self):
        index = np.where(self.board == 0)
        if index[1] == 2:
            return False
        move_to = (index[0], index[1]+1)
        self.swap(index, move_to)
        return True

    def move_right(self):
        index = np.where(self.board == 0)
        if index[1] == 0:
            return False
        move_to = (index[0], index[1]-1)
        self.swap(index, move_to)
        return True

    def move_up(self):
        index = np.where(self.board == 0)
        if index[0] == 2:
            return False
        move_to = (index[0]+1, index[1])
        self.swap(index, move_to)
        return True

    def move_down(self):
        index = np.where(self.board == 0)
        if index[0] == 0:
            return False
        move_to = (index[0]-1, index[1])
        self.swap(index, move_to)
        return True

    def __str__(self):
        result = ''
        for i in range(3):
            result += '| '
            for j in range(3):
                result += '%s | ' % (str(int(self.board[i, j])) if self.board[i, j] != 0 else ' ')
            result += '\n'
        return result
    
    def shuffle(self):
        for i in range(randint(10, 100)):
            f = choice([
                self.move_down,
                self.move_up,
                self.move_right,
                self.move_left
            ])
            f()

    def apply_chain(self, chain, with_display=False):
        chain_map = {
            'cima': self.move_up,
            'baixo': self.move_down,
            'esquerda': self.move_left,
            'direita': self.move_right,
        }
        for ch in chain:
            chain_map[ch]()
            if with_display:
                print(self)

    def cost(self):
        reference = {
            0: (0, 0),
            1: (0, 1),
            2: (0, 2),

            3: (1, 0),
            4: (1, 1),
            5: (1, 2),

            6: (2, 0),
            7: (2, 1),
            8: (2, 2),
        }

        error = 0.0
        for i in range(9):
            index = np.where(self.board == i)
            ref_index = reference[i]
            error += (index[0] - ref_index[0]) ** 2 + (index[1] - ref_index[1]) ** 2
        error /= 9.0
        return error[0]


class Chromosome:
    VALID_MOVES = ['cima', 'baixo', 'esquerda', 'direita']

    def __init__(self, puzzle, gene=None):
        if gene is None:
            gene = []

        self.error = None
        self.error_puzzle_cost = None
        self.error_gene_len = None
        self.puzzle = puzzle
        self.gene = gene
        self.update_error()

    def update_error(self):
        temp = deepcopy(self.puzzle)
        temp.apply_chain(self.gene)

        self.error_puzzle_cost = temp.cost()
        self.error_gene_len = len(self.gene) * 0.01
        self.error = self.error_puzzle_cost + self.error_gene_len

    @staticmethod
    def cross_over(a, b):
        if len(b.gene) > len(a.gene):
            return Chromosome.cross_over(b, a)

        geneA = []
        geneB = []

        len_a = len(a.gene)
        len_b = len(b.gene)

        for i in range(len_b):
            if random.random() < 0.5:
                geneA.append(a.gene[i])
                geneB.append(b.gene[i])
            else:
                geneA.append(b.gene[i])
                geneB.append(a.gene[i])

        if len_b != len_a:
            for i in range(len_a-len_b):
                geneA.append(a.gene[len_b+i])

        return Chromosome(a.puzzle, geneA), Chromosome(b.puzzle, geneB)

    def mutate(self, allow_only_growing=False):
        add_vs_mutate_chance = 0.5
        if not self.gene:
            add_vs_mutate_chance = 1.0

        if random.random() < add_vs_mutate_chance:
            self.gene.append(random.choice(self.VALID_MOVES))
        else:
            i = random.randint(0, len(self.gene)-1)
            self.gene[i] = random.choice(self.VALID_MOVES)

    def __str__(self):
        return '(%d)  %s' % (len(self.gene), ' -> '.join(self.gene))


class GeneticsSolver:
    def __init__(self, puzzle, initial_population=1000, mutation_chance=0.9, cross_over_rate=0.3):
        self.puzzle = puzzle

        # parameters
        self.initial_population = initial_population
        self.mutation_chance = mutation_chance
        self.cross_over_rate = cross_over_rate

        # variables
        self.population = []
        self.best = None
        self.error = None

    def _init_population(self):
        self.population = [Chromosome(puzzle=self.puzzle) for _ in range(self.initial_population)]

    def _calculate_error(self):
        if not self.population:
            return

        if self.best is None:
            self.best = self.population[0]

        error = 0
        for p in self.population:
            p.update_error()
            if p.error < self.best.error:
                self.best = deepcopy(p)
            error += p.error
        self.error = error / len(self.population)

    def _select_bests(self):
        totals = []
        running_total = 0

        for p in self.population:
            w = 1 / (0.000001 + p.error)
            running_total += w
            totals.append(running_total)

        def select_i():
            rnd = random.random() * running_total
            for i, total in enumerate(totals):
                if rnd < total:
                    return i

        result = []
        while len(result) < self.initial_population:
            i = select_i()
            result.append(self.population[i])
        self.population = result

    def _cross_over(self, always=False):
        if len(self.population) < 2:
            return

        cross_over_occur = math.ceil(len(self.population) * self.cross_over_rate)
        list_of_fertile_genes = [i for i in range(len(self.population))]

        for i in range(int(cross_over_occur)):
            a, b = random.sample(list_of_fertile_genes, 2)
            for idx, k in enumerate(list_of_fertile_genes):
                if k == b or k == a:
                    del list_of_fertile_genes[idx]

            offspring_a, offspring_b = Chromosome.cross_over(self.population[a], self.population[b])
            if (offspring_a.error < min(self.population[a].error, self.population[b].error) and \
               offspring_b.error < min(self.population[a].error, self.population[b].error)) or always:
                self.population[a] = offspring_a
                self.population[b] = offspring_b

    def _mutate(self):
        for i in self.population:
            if random.random() < self.mutation_chance:
                i.mutate(True)

    def solve(self, max_iter=1000, optimal_error=0):
        random.seed()
        self._init_population()

        iteration = 0
        while iteration < max_iter and (not self.best or self.best.error_puzzle_cost > optimal_error):
            self._cross_over(always=True)
            self._calculate_error()
            self._select_bests()
            self._mutate()

            self._display(iteration)
            iteration += 1

        return self.best

    def _display(self, iteration):
        print('~~~~~~~~ iteração: %d ~~~~~~~~' % iteration)
        print('população: Tamanho(%d)   Erro Total(%0.4f)' % (
            len(self.population),
            self.error
        ))

        if self.best:
            print('melhor: %0.4f   ->   %0.4f  +  %0.4f' % (
                self.best.error,
                self.best.error_puzzle_cost,
                self.best.error_gene_len
            ))
            print('melhor é: %s' % self.best)
            print('--------------------------------')


    
TAXA_MUTACAO = 0.9
TAXA_CROSSOVER = 0.3
TAMANHO_POPULACAO = 1000
ITERACAO_MAX = 1000

try:
    p = Tabuleiro()
    p.shuffle()

    gSolver = GeneticsSolver(p, TAMANHO_POPULACAO, TAXA_MUTACAO, TAXA_CROSSOVER)
    gSolver.solve(max_iter=ITERACAO_MAX)

    print('\n\n====================================================')
    print('Original:')
    print(p)
    print('Solução encontrada:')
    print(gSolver.best.gene)
    p.apply_chain(gSolver.best.gene, with_display=True)
    print('Resultado:')
    print(p)
except KeyboardInterrupt:
    pass

In [ ]:
"""
Navegação em Grafo - Algoritmo Genético

necessário instalar a biblioteca networkx
"""

import random
import networkx as nx
import matplotlib.pyplot as plt

graph = {
    'A': {'B': 2, 'C': 3, 'D': 4},
    'B': {'A': 2, 'E': 5, 'F': 6},
    'C': {'A': 3, 'G': 7, 'H': 8},
    'D': {'A': 4, 'I': 9, 'J': 10},
    'E': {'B': 5, 'K': 11, 'L': 12},
    'F': {'B': 6, 'M': 13, 'N': 14},
    'G': {'C': 7},
    'H': {'C': 8},
    'I': {'D': 9},
    'J': {'D': 10},
    'K': {'E': 11},
    'L': {'E': 12},
    'M': {'F': 13},
    'N': {'F': 14}
}

# Recebe um grafo, um nó inicial e um nó final
# Retorna uma rota aleatória do início ao fim ou Nenhuma se não for encontrada
def generateRoute(graph, start, finish):
    visited = set()
    route = [start]
    while route[-1] != finish:
        currentNode = route[-1]
        visited.add(currentNode)
        choices = [node for node in graph[currentNode] if node not in visited]

        if not choices:
            return None
        
        nextNode = random.choice(choices)
        route.append(nextNode)

    return route

# Recebe um gráfo e uma rota
# Retorna o custo de travessia
def fitness(graph, route):
    cost = 0

    for i in range(len(route) - 1):
        currentNode = route[i]
        nextNode = route[i+1]
        cost += graph[currentNode][nextNode]
    return cost

# Recebe uma população
# Retorna a tupla que contém a melhor rota juntamente com o seu custo
def findBest(population):
    best = list(population)[0]
    for route in population:
        if route[1] < best[1]:
            best = route
    
    return best

# Tem em conta o tamanho da população, um gráfo e o máximo de tentativas
# Retorna uma população de rotas através do gráfo,
# tentará atingir o tamanho da população até atingir o máximo de tentativas
def generatePopulation(populationSize, graph, maxAttempts):
    population = []
    routes = set()
    iteration = 0
    while len(population) < populationSize:
        if len(population) == populationSize or iteration == maxAttempts:
            break

        route = generateRoute(graph, start, finish)

        if route is not None and tuple(route) not in routes:
            population.append((route, fitness(graph, route)))
            routes.add(tuple(route))

        iteration += 1

    return population

# Recebe uma rota e um gráfo
# Retorna True se a rota existir
def validateRoute(route, graph):
    for i in range(len(route) - 1):
        currentNode = route[i]
        nextNode = route[i+1]
        if nextNode not in graph[currentNode]:
            return False
        
    return True

# Recebe uma rota e uma hipótese de mutação
# Retorna rota mutada
def mutateRoute(graph, route, chance):
    while True:
        for i in range(1, len(route) - 1):
            randInt = random.randint(0, 100)
            currentNode = route[i]
            previousNode = route[i - 1]
            nextNode = route[i + 1]
            finish = route[-1]

            if randInt <= chance*100 and currentNode is not finish:
                availableNodes = [node for node in graph[previousNode] if node not in route and nextNode in graph[node]]

                if len(availableNodes) == 0:
                    continue
                else:
                    mutatedNode = random.choice(availableNodes)
                    route[i] = mutatedNode

        if validateRoute(route, graph):
            break

        print("Não válido")

    return route

def mutatePopulation(population, graph, chance):
    mutatedPopulation = []
    for tuple in population:
        route = tuple[0]
        mutated = mutateRoute(graph, route, chance)
        mutatedFitness = fitness(graph, mutated)
        currentFitness = fitness(graph, route)
        if mutatedFitness < currentFitness:
            mutatedPopulation.append((mutated, mutatedFitness))
        else:
            mutatedPopulation.append(tuple)

    return mutatedPopulation

    
start = 'A'
finish = 'N'
start = input("Nó inicial: ").upper()
finish = input("Nó objetivo: ").upper()

population = generatePopulation(10, graph, 1000)

if len(population) == 0:
    print(">Caminho não encontrado!")
else:
    previousBest = findBest(population)
    attempts = 10

    # Define o caminho que pretende destacar (representado como uma lista de nós)
    path = previousBest[0]
    print("==================================================")
    while True:   
        best = findBest(population)
        print(">Best:")
        print(">" + str(best))
        print("==================================================")
    
        population = mutatePopulation(population, graph, 0.25)
        previousBest = best
        path = best[0]
        
        if best[1] == previousBest[1]:
            attempts -= 1
        else:
            attempts = 10
        
        if attempts == 0:
            break
    
    print()
    print(">Encontrado!")
    
    # Cria o grafo direcionado
    G = nx.DiGraph(graph)
    
    # Encontra os nós no caminho
    edges_in_path = [(path[i], path[i+1]) for i in range(len(path)-1)]
    
    # Cria um dicionário para armazenar cores de nós e arestas
    colors = {}
    for node in G.nodes():
        if node in path:
            colors[node] = 'green'
        else:
            colors[node] = 'red'
    for edge in G.edges():
        if edge in edges_in_path:
            colors[edge] = 'green'
        else:
            colors[edge] = 'black'
    
    # Desenha o gráfico com as cores atualizadas
    nx.draw(G, with_labels=True, node_color=[colors[node] for node in G.nodes()], 
            edge_color=[colors[edge] for edge in G.edges()], 
            width=2, font_size=16, font_weight='bold')
    plt.show()

In [ ]:
"""
Usar um algoritmo genético para optimizar a poupança numa lista de compras
"""

from random import randint
import random

#funcao ordnear ordem decrescente
def decres1(count, vet_qualquer=[],vet2=[],vet3=[],vet4=[]):
 count=0
 for t in range (0,8):
  xvet_qualquer=vet_qualquer[count:count+8]  
  xvet2=vet2[count:count+8]
  xvet3=vet3[count:count+8]
  for s in range (0,10):    
   for x in range (0,7):           
      if xvet_qualquer[x+1]>xvet_qualquer[x]:
       aux=xvet_qualquer[x]
       xvet_qualquer[x]=xvet_qualquer[x+1]
       xvet_qualquer[x+1]=aux
       aux1=xvet2[x]     
       xvet2[x]=xvet2[x+1]
       xvet2[x+1]=aux1
       aux2=xvet3[x]
       xvet3[x]=xvet3[x+1]
       xvet3[x+1]=aux2
  vet_qualquer[count:count+8]=xvet_qualquer
  vet2[count:count+8]=xvet2
  vet4.append(xvet3)
  count=count+8     
      
def decres2(vet_qualquer=[],vet2=[]):      
  for s in range (0,200):    
   for x in range (0,198):           
      if vet_qualquer[x+1]>vet_qualquer[x]:
       aux=vet_qualquer[x]
       vet_qualquer[x]=vet_qualquer[x+1]
       vet_qualquer[x+1]=aux  
       aux1=vet2[x]     
       vet2[x]=vet2[x+1]
       vet2[x+1]=aux1  


#matriz produtos
minimo=[]
z=[]
funcao_z=[]
lista_somaz=[]
soma=0
soma_z=0
lista_pedidos=[]
lista_beneficio=[]
#restricao por produto
r=[10,15,20,25,30,35,40,45]
#restricao por supermercado
s=[2,4,5,6,7,8,9,9]

#matriz custo
custo=matriz8x8=[
                 [3.99,4.99,4.75,2.99,3.99,3.99,1.97,4.98],

                 [5.99,12.90,5.98,11.99,9.49,6.99,15.77,13.98],

                 [12.90,9.39,9.90,15.99,14.59,12.99,9.99,10.84],

                 [0.99,4.99,1.18,1.79,0.99,1.99,1.18,3.98],

                 [3.99,6.89,7.98,9.99,7.89,6.99,5.97,6.88],

                 [5.99,15.99,0,3.49,10.09,6.99,5.97,8.98],

                 [12.90,7.59,6.98,7.99,4.79,9.99,9.75,10.38],

                 [4.99,7.98,4.98,7.99,7.89,5.99,5.98,3.58]
                 ]

#matrizes pedidos, e beneficios
beneficio=matriz8x8=[[0 for col in range(8)] for row in range(8)]
pedidos=matriz8x8=[[0 for col in range(8)] for row in range(8)]



for x in range (0,8):
     minimo.append(min(custo[x]))
     
# preenchendo a lista beneficio    
for x in range (0,8):
    for y in range (0,8):
        beneficio[x][y]=minimo[x]-custo[x][y]
        lista_beneficio.append(beneficio[x][y])
        

#algoritmo genetico
TM=0.01
TC=0.85
tam=64
lista_z=[]
indiv=[]
I=[[]]*20200
F=[[]]*200
Z=[[]]*200
Beneficio=[[]]*200
B1=[]
z=0
w=tam
soma_beneficio=[0]*200
tamanho=[]

#gerando individuos
for t in range (0,200):
 for x in range (0,8):
    soma=0
    for y in range (0,8):
        n=randint(0,s[x])
        pedidos[x][y]=n
        soma=pedidos[x][y]+soma
        if soma>r[x]:
           pedidos[x][y]=0
           soma=soma-pedidos[x][y]
        indiv.append(pedidos[x][y])     
         

#separando individuos
for x in range (0,200):
 I[x]=indiv[z:w]
 z=z+tam
 w=w+tam

 
#verificando a taxa de cruzamento e de mutação
t=0  
count=0  
z=199
k=1
while t<20000:
  n=random.random()
  if n<TC:
      s=randint(1,62)
      I[t+200]=I[t][0:s]+I[t+1][s:64]
      I[t+201]=I[t+1][0:s]+I[t][s:64]
  else:
        I[t+200]=I[t]
        I[t+201]=I[t+1]
  m=random.random()
  if m<TM:
   j=randint(0,63)
   n1=randint(0,10)
   n2=randint(0,10)
   I[t+200][j]=n1
   I[t+201][j]=n2      
  t=t+2

#os filhos resultantes de 100 geraões de cruzamentos e mutações
F=I[20000:20200]  

#a lista função objetivo, pedidos * beneficio
count=0
for t in range (0,200):
 soma_z=0   
 for x in range (0,tam):
         funcao_z.append(F[t][x]*lista_beneficio[x])
         soma_z=funcao_z[x+count]+soma_z
 count=64+count        
 lista_somaz.append(soma_z)

z=0
w=tam
#criando individuo da funcao z
for x in range (0,200):
 Z[x]=funcao_z[z:w]
 z=z+tam
 w=w+tam 
 
z=0
w= 8
#ordenando ordem crescente por funcao z
for t in range (0,200):
 decres1(0,Z[t], F[t], lista_beneficio,B1)
 Beneficio[t]=B1[z:w]
 z=z+8
 w=w+8    

   
lista_somaz1=[]   
#penalizando  os individuos
for t in range (0,200):
     count = 0
     somaz1=0
     for x in range (0,8):
         soma=0
         count1=0
         for y in range (count, count+8):              
               soma=F[t][y]+soma
               if soma>r[x]:
                   resultado=soma-r[x]
                   F[t][y]=F[t][y]-resultado
                   Z[t][y]=F[t][y]*Beneficio[t][x][count1]
                   soma=r[x]
               count1=count1+1
               somaz1=Z[t][y]+somaz1
         count=count+8
     lista_somaz1.append(somaz1)    

for t in range (0,200):
    tamanho.append(t)


#ordenando em ordem crescente a lista que soma o custo total de um individuo
decres2(lista_somaz1, tamanho)

print('Solucao para o problema' , 'Individuo', F[tamanho[0]])
print('Custo da compra: ',lista_somaz1[0]*-1,'Euros')
print('Economia de',max(lista_somaz1)-min(lista_somaz1) ,'Euros em relacao ao maior gasto.') 